In [ ]:

# T-MLP PROJECT STRUCTURE AUDIT

from pathlib import Path
import os
import ast
import json
import zipfile
import importlib.metadata as metadata

PROJECT_ROOT = Path("/content/drive/MyDrive/T-MLP")

# Optional Drive mount if it is not already mounted.
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not PROJECT_ROOT.exists():
    print(f"PROJECT ROOT NOT FOUND: {PROJECT_ROOT}")
    print("Available folders under MyDrive:")
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        for p in sorted(drive_root.iterdir()):
            print(" ", p.name, "/" if p.is_dir() else "")
    raise FileNotFoundError(
        f"Please update PROJECT_ROOT to the actual project directory."
    )

SKIP_DIRS = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "node_modules", ".venv", "venv"
}

def section(title):
    print("\n" + "=" * 85)
    print(title)
    print("=" * 85)

def human_size(num_bytes):
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.2f} {unit}"
        size /= 1024

def safe_file_size(path):
    try:
        return path.stat().st_size
    except OSError:
        return 0

# 1. DIRECTORY TREE
section("1. PROJECT DIRECTORY TREE")

MAX_DEPTH = 4
MAX_ENTRIES_PER_DIR = 100

def print_tree(folder, prefix="", depth=0):
    if depth > MAX_DEPTH:
        return

    try:
        entries = sorted(
            [p for p in folder.iterdir()
             if p.name not in SKIP_DIRS],
            key=lambda p: (not p.is_dir(), p.name.lower())
        )
    except (PermissionError, OSError) as exc:
        print(prefix + f"[Cannot read directory: {exc}]")
        return

    for i, path in enumerate(entries[:MAX_ENTRIES_PER_DIR]):
        last = i == min(len(entries), MAX_ENTRIES_PER_DIR) - 1
        connector = "└── " if last else "├── "
        suffix = "/" if path.is_dir() else (
            f"  [{human_size(safe_file_size(path))}]"
        )
        print(prefix + connector + path.name + suffix)

        if path.is_dir() and depth < MAX_DEPTH:
            child_prefix = prefix + ("    " if last else "│   ")
            print_tree(path, child_prefix, depth + 1)

    if len(entries) > MAX_ENTRIES_PER_DIR:
        print(prefix + f"... {len(entries) - MAX_ENTRIES_PER_DIR} entries omitted")

print("Root:", PROJECT_ROOT)
print_tree(PROJECT_ROOT)

# 2. FILE INVENTORY BY EXTENSION AND DIRECTORY
section("2. FILE INVENTORY")

extension_counts = {}
directory_counts = {}
important_files = []

for current, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    current_path = Path(current)

    directory_counts[str(current_path.relative_to(PROJECT_ROOT))] = len(files)

    for filename in files:
        path = current_path / filename
        ext = path.suffix.lower() or "[no extension]"
        extension_counts[ext] = extension_counts.get(ext, 0) + 1

        if ext in {
            ".py", ".ipynb", ".yaml", ".yml", ".json",
            ".toml", ".txt", ".md", ".keras", ".h5",
            ".npz", ".parquet", ".csv", ".wav", ".flac"
        }:
            important_files.append(path)

print("\nFile counts by extension:")
for ext, count in sorted(extension_counts.items()):
    print(f"  {ext:18} {count:>8}")

print("\nFiles per directory:")
for directory, count in sorted(directory_counts.items()):
    print(f"  {directory or '.':55} {count:>8}")

# 3. PYTHON SOURCE STRUCTURE
section("3. PYTHON SCRIPT STRUCTURE")

python_files = [
    p for p in PROJECT_ROOT.rglob("*.py")
    if not any(part in SKIP_DIRS for part in p.parts)
]

for path in sorted(python_files):
    print(f"\n--- {path.relative_to(PROJECT_ROOT)} ---")
    try:
        source = path.read_text(encoding="utf-8")
        tree = ast.parse(source)
        print(f"Lines: {len(source.splitlines())}")

        imports = []
        definitions = []
        assignments = []

        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                imports.extend(alias.name for alias in node.names)
            elif isinstance(node, ast.ImportFrom):
                imports.append(
                    f"from {node.module or ''} import "
                    + ", ".join(alias.name for alias in node.names)
                )

        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                args = [arg.arg for arg in node.args.args]
                definitions.append(
                    f"function {node.name}({', '.join(args)})"
                )
            elif isinstance(node, ast.ClassDef):
                definitions.append(f"class {node.name}")
            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = (
                    node.targets if isinstance(node, ast.Assign)
                    else [node.target]
                )
                for target in targets:
                    if isinstance(target, ast.Name):
                        assignments.append(target.id)

        print("Top-level definitions:")
        for item in definitions:
            print("  ", item)

        print("Top-level variable names:")
        print("  ", ", ".join(assignments[:40]) or "(none)")

        print("Imports:")
        for item in sorted(set(imports))[:35]:
            print("  ", item)
        if len(set(imports)) > 35:
            print("  ... additional imports omitted")

    except Exception as exc:
        print("Could not parse:", repr(exc))

# 4. MODEL CHECKPOINTS
section("4. MODEL CHECKPOINTS")

model_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".keras", ".h5", ".hdf5", ".ckpt"}
    and not any(part in SKIP_DIRS for part in p.parts)
]

if not model_files:
    print("No model checkpoints found under the project root.")

for path in sorted(model_files):
    print(f"\n{path.relative_to(PROJECT_ROOT)}")
    print("  Size:", human_size(safe_file_size(path)))

    # Inspect Keras ZIP metadata without loading model weights.
    if path.suffix.lower() == ".keras":
        try:
            with zipfile.ZipFile(path, "r") as archive:
                names = archive.namelist()
                print("  ZIP integrity:", "OK" if archive.testzip() is None
                      else "ERROR")
                print("  Archive entries:", names[:15])

                if "metadata.json" in names:
                    meta = json.loads(archive.read("metadata.json"))
                    print("  Metadata:", json.dumps(meta, indent=2)[:1500])

                if "config.json" in names:
                    config = json.loads(archive.read("config.json"))
                    print("  Model name:",
                          config.get("config", {}).get("name"))
                    print("  Top-level class:",
                          config.get("class_name"))

                    layers = (
                        config.get("config", {}).get("layers", [])
                    )
                    if layers:
                        print("  Configured layers:")
                        for layer in layers:
                            cfg = layer.get("config", {})
                            print(
                                "   -",
                                cfg.get("name", "?"),
                                "|",
                                layer.get("class_name", "?")
                            )
        except Exception as exc:
            print("  Archive inspection error:", repr(exc))

# 5. DATASET SPLIT INVENTORY
section("5. DATASET DIRECTORIES")

candidate_data_dirs = [
    PROJECT_ROOT / "processed_data",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "dataset",
    PROJECT_ROOT / "datasets",
]

# Also discover likely data directories elsewhere in the project.
for path in PROJECT_ROOT.rglob("*"):
    if not path.is_dir():
        continue
    if any(part in SKIP_DIRS for part in path.parts):
        continue
    if path.name.lower() in {
        "train", "validation", "valid", "val", "test",
        "processed_data", "datasets", "dataset"
    }:
        if path not in candidate_data_dirs:
            candidate_data_dirs.append(path)

for folder in sorted(set(candidate_data_dirs), key=str):
    if not folder.is_dir():
        continue

    print(f"\n{folder.relative_to(PROJECT_ROOT)}")
    files = [
        p for p in folder.rglob("*")
        if p.is_file()
        and not any(part in SKIP_DIRS for part in p.parts)
    ]

    counts = {}
    total_bytes = 0
    for p in files:
        counts[p.suffix.lower() or "[no extension]"] = (
            counts.get(p.suffix.lower() or "[no extension]", 0) + 1
        )
        total_bytes += safe_file_size(p)

    print("  Total files:", len(files))
    print("  Total size:", human_size(total_bytes))
    print("  Formats:", counts)

    # Inspect one NPZ file's keys and array shapes without loading all files.
    sample_npz = next((p for p in files if p.suffix.lower() == ".npz"), None)
    if sample_npz:
        try:
            import numpy as np
            with np.load(sample_npz, allow_pickle=False) as sample:
                print("  Example NPZ:", sample_npz.name)
                for key in sample.files:
                    arr = sample[key]
                    print(
                        f"    {key}: shape={arr.shape}, "
                        f"dtype={arr.dtype}"
                    )
        except Exception as exc:
            print("  NPZ sample inspection failed:", repr(exc))

# 6. NOTEBOOK INVENTORY
section("6. NOTEBOOKS")

notebooks = [
    p for p in PROJECT_ROOT.rglob("*.ipynb")
    if not any(part in SKIP_DIRS for part in p.parts)
]

for path in sorted(notebooks):
    print(f"\n{path.relative_to(PROJECT_ROOT)}")
    print("  Size:", human_size(safe_file_size(path)))
    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
        cells = notebook.get("cells", [])
        code_cells = sum(c.get("cell_type") == "code" for c in cells)
        markdown_cells = sum(c.get("cell_type") == "markdown" for c in cells)
        print(
            f"  Cells: {len(cells)} "
            f"(code={code_cells}, markdown={markdown_cells})"
        )

        # Show markdown headings and code-cell first lines only.
        shown = 0
        for cell in cells:
            source = "".join(cell.get("source", [])).strip()
            if not source:
                continue

            if cell.get("cell_type") == "markdown":
                headings = [
                    line for line in source.splitlines()
                    if line.strip().startswith("#")
                ]
                for heading in headings[:3]:
                    print("   ", heading[:140])
                    shown += 1

            elif cell.get("cell_type") == "code":
                first_line = source.splitlines()[0]
                print("    CODE:", first_line[:140])
                shown += 1

            if shown >= 20:
                print("    ... notebook summary truncated")
                break
    except Exception as exc:
        print("  Notebook inspection failed:", repr(exc))

# 7. RUNTIME VERSIONS
section("7. RUNTIME ENVIRONMENT")

for package in [
    "tensorflow", "tf-keras", "keras",
    "tensorflow-model-optimization", "numpy",
    "scipy", "soundfile", "scikit-learn"
]:
    try:
        print(f"{package:35} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:35} NOT INSTALLED")

# 8. IMPORTANT FILES AT A GLANCE
section("8. IMPORTANT PROJECT FILES")

keywords = (
    "train", "model", "qat", "tmlp", "dataset",
    "preprocess", "evaluate", "config", "metric",
    "inference", "quant", "loss"
)

for path in sorted(important_files):
    if any(word in path.name.lower() for word in keywords):
        print(
            f"{str(path.relative_to(PROJECT_ROOT)):75} "
            f"{human_size(safe_file_size(path))}"
        )

section("AUDIT COMPLETE")
print("No project scripts were executed by this audit.")
print("No model was trained and no project files were modified.")



1. PROJECT DIRECTORY TREE
Root: /content/drive/MyDrive/T-MLP
├── data_pipeline/
│   ├── data_preprocessing.py  [2.41 KB]
│   ├── dataset.py  [8.16 KB]
│   ├── inspect_data.py  [2.15 KB]
│   └── mixing.py  [3.26 KB]
├── processed_data/
│   ├── test/
│   │   ├── sample_000000.npz  [259.26 KB]
│   │   ├── sample_000001.npz  [492.86 KB]
│   │   ├── sample_000002.npz  [609.05 KB]
│   │   ├── sample_000003.npz  [172.60 KB]
│   │   ├── sample_000004.npz  [596.96 KB]
│   │   ├── sample_000005.npz  [556.06 KB]
│   │   ├── sample_000006.npz  [816.32 KB]
│   │   ├── sample_000007.npz  [446.96 KB]
│   │   ├── sample_000008.npz  [763.16 KB]
│   │   ├── sample_000009.npz  [226.74 KB]
│   │   ├── sample_000010.npz  [907.31 KB]
│   │   ├── sample_000011.npz  [440.45 KB]
│   │   ├── sample_000012.npz  [1.41 MB]
│   │   ├── sample_000013.npz  [321.43 KB]
│   │   ├── sample_000014.npz  [236.98 KB]
│   │   ├── sample_000015.npz  [248.83 KB]
│   │   ├── sample_000016.npz  [739.14 KB]
│   │   ├── sample_00

In [3]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/T-MLP")
OUTPUT = ROOT / "source_inspection.txt"

files_to_inspect = [
    ROOT / "data_pipeline/data_preprocessing.py",
    ROOT / "data_pipeline/mixing.py",
    ROOT / "data_pipeline/dataset.py",
]

with OUTPUT.open("w", encoding="utf-8") as out:
    for path in files_to_inspect:
        out.write(f"\n\n{'=' * 90}\n")
        out.write(f"FILE: {path.relative_to(ROOT)}\n")
        out.write(f"{'=' * 90}\n\n")
        out.write(path.read_text(encoding="utf-8"))

    evaluator = ROOT / "evaluation/evaluation_all_models.py"

    if evaluator.exists():
        lines = evaluator.read_text(encoding="utf-8").splitlines()

        # Configuration, preprocessing, prediction, and evaluation sections
        sections = [
            (1, 180),
            (400, 570),
            (570, 850),
            (1450, 1610),
        ]

        for start, end in sections:
            out.write(f"\n\n{'=' * 90}\n")
            out.write(
                f"FILE: evaluation/evaluation_all_models.py "
                f"(lines {start}-{min(end, len(lines))})\n"
            )
            out.write(f"{'=' * 90}\n\n")
            out.write("\n".join(lines[start - 1:end]))

print(f"Inspection file created: {OUTPUT}")
print(f"Size: {OUTPUT.stat().st_size / 1024:.1f} KB")

Inspection file created: /content/drive/MyDrive/T-MLP/source_inspection.txt
Size: 13.8 KB


In [4]:
from pathlib import Path
import json
import zipfile

ROOT = Path("/content/drive/MyDrive/T-MLP")
MODEL_PATH = ROOT / "saved_models" / "TMLP_best.keras"

print("Model exists:", MODEL_PATH.exists())

if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)

with zipfile.ZipFile(MODEL_PATH, "r") as archive:
    config = json.loads(archive.read("config.json"))

    print("\nModel name:", config.get("config", {}).get("name"))
    print("Model class:", config.get("class_name"))

    print("\nSaved architecture:")
    for layer in config.get("config", {}).get("layers", []):
        print(
            f"{layer.get('name', '?'):30} "
            f"{layer.get('class_name', '?'):25} "
            f"{layer.get('config', {}).get('batch_input_shape', layer.get('config', {}).get('shape', ''))}"
        )

print("\nSaved models:")
for path in sorted((ROOT / "saved_models").glob("*")):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / 1024:.2f} KB")

print("\nDataset counts:")
for split in ["train", "validation", "test"]:
    folder = ROOT / "processed_data" / split
    print(f"{split}: {len(list(folder.glob('*.npz')))} samples")

Model exists: True

Model name: T_MLP
Model class: Functional

Saved architecture:
noisy_audio                    InputLayer                
temporal_reshape               Reshape                   
feature_projection             Dense                     
temporal_mlp_block_1           TemporalMLPBlock          
temporal_mlp_block_2           TemporalMLPBlock          
feature_reduction              Dense                     
output_projection              Dense                     
output_waveform                Reshape                   

Saved models:
GRU_best.keras: 478.09 KB
MLP_best.keras: 3876.92 KB
TMLP_QAT_best.weights.h5: 392.95 KB
TMLP_QAT_best_before_final.weights.h5: 392.95 KB
TMLP_QAT_final_best.weights.h5: 392.95 KB
TMLP_QAT_initial.weights.h5: 222.52 KB
TMLP_QAT_int8.tflite: 54.09 KB
TMLP_best.keras: 292.70 KB

Dataset counts:
train: 800 samples
validation: 100 samples
test: 100 samples


In [5]:
from pathlib import Path
import json
import zipfile
import h5py

ROOT = Path("/content/drive/MyDrive/T-MLP")
MODEL_PATH = ROOT / "saved_models" / "TMLP_best.keras"

with zipfile.ZipFile(MODEL_PATH, "r") as archive:
    config = json.loads(archive.read("config.json"))

    print("=" * 70)
    print("MODEL INPUT / OUTPUT CONFIGURATION")
    print("=" * 70)

    for layer in config["config"]["layers"]:
        name = layer.get("name", "?")
        cls = layer.get("class_name", "?")
        cfg = layer.get("config", {})

        if cls in {"InputLayer", "Reshape", "Dense"}:
            print(f"\n{name} ({cls})")
            for key in (
                "batch_shape", "batch_input_shape", "shape",
                "target_shape", "units", "activation", "use_bias"
            ):
                if key in cfg:
                    print(f"  {key}: {cfg[key]}")

    print("\n" + "=" * 70)
    print("SAVED WEIGHT STRUCTURE")
    print("=" * 70)

    with archive.open("model.weights.h5") as weight_file:
        # h5py needs a filesystem path, so copy the archive member to memory.
        import io
        with h5py.File(io.BytesIO(weight_file.read()), "r") as h5:
            def show_item(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f"{name}: shape={obj.shape}, dtype={obj.dtype}")

            h5.visititems(show_item)

MODEL INPUT / OUTPUT CONFIGURATION

noisy_audio (InputLayer)
  batch_shape: [None, 256]

temporal_reshape (Reshape)
  target_shape: [256, 1]

feature_projection (Dense)
  units: 64
  activation: relu
  use_bias: True

feature_reduction (Dense)
  units: 32
  activation: relu
  use_bias: True

output_projection (Dense)
  units: 1
  activation: linear
  use_bias: True

output_waveform (Reshape)
  target_shape: [256]

SAVED WEIGHT STRUCTURE
layers/dense/vars/0: shape=(1, 64), dtype=float32
layers/dense/vars/1: shape=(64,), dtype=float32
layers/dense_1/vars/0: shape=(64, 32), dtype=float32
layers/dense_1/vars/1: shape=(32,), dtype=float32
layers/dense_2/vars/0: shape=(32, 1), dtype=float32
layers/dense_2/vars/1: shape=(1,), dtype=float32
layers/temporal_mlp_block/dense_1/vars/0: shape=(64, 64), dtype=float32
layers/temporal_mlp_block/dense_1/vars/1: shape=(64,), dtype=float32
layers/temporal_mlp_block/dense_2/vars/0: shape=(64, 64), dtype=float32
layers/temporal_mlp_block/dense_2/vars/1: sh

In [6]:
from pathlib import Path
import json
import zipfile

model_path = Path(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras"
)

with zipfile.ZipFile(model_path, "r") as archive:
    config = json.loads(archive.read("config.json"))

for layer in config["config"]["layers"]:
    if layer.get("class_name") == "TemporalMLPBlock":
        print(f"\nLayer: {layer.get('name')}")
        print(json.dumps(layer.get("config", {}), indent=2))

        print("Saved inbound connections:")
        print(json.dumps(layer.get("inbound_nodes", []), indent=2))

print("\nModel input/output:")
print("Input:", config["config"].get("input_layers"))
print("Output:", config["config"].get("output_layers"))


Layer: temporal_mlp_block_1
{
  "name": "temporal_mlp_block_1",
  "hidden_dim": 64,
  "dropout_rate": 0.1,
  "trainable": true,
  "dtype": {
    "module": "keras",
    "class_name": "DTypePolicy",
    "config": {
      "name": "float32"
    },
    "registered_name": null,
    "shared_object_id": 139473906225168
  }
}
Saved inbound connections:
[
  {
    "args": [
      {
        "class_name": "__keras_tensor__",
        "config": {
          "shape": [
            null,
            256,
            64
          ],
          "dtype": "float32",
          "keras_history": [
            "feature_projection",
            0,
            0
          ]
        }
      }
    ],
    "kwargs": {}
  }
]

Layer: temporal_mlp_block_2
{
  "name": "temporal_mlp_block_2",
  "hidden_dim": 64,
  "dropout_rate": 0.1,
  "trainable": true,
  "dtype": {
    "module": "keras",
    "class_name": "DTypePolicy",
    "config": {
      "name": "float32"
    },
    "registered_name": null,
    "shared_object_id":

In [7]:
from pathlib import Path
import zipfile
import h5py
import numpy as np
import io
import keras
from keras import layers

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
project_dir = Path("/content/drive/MyDrive/T-MLP")
model_path = project_dir / "saved_models" / "TMLP_best.keras"

assert model_path.exists(), f"Checkpoint not found: {model_path}"

# --------------------------------------------------
# 2. Rebuild the original custom block
# --------------------------------------------------
@keras.saving.register_keras_serializable(package="TMLP")
class TemporalMLPBlockRestore(layers.Layer):
    def __init__(self, hidden_dim=64, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout_rate

        self.dense_1 = layers.Dense(
            hidden_dim, activation="relu", name="dense_1"
        )
        self.dropout = layers.Dropout(dropout_rate, name="dropout")
        self.dense_2 = layers.Dense(
            hidden_dim, activation=None, name="dense_2"
        )
        self.add = layers.Add(name="residual_add")
        self.layer_norm = layers.LayerNormalization(name="layer_norm")

    def call(self, inputs, training=None):
        x = self.dense_1(inputs)
        x = self.dropout(x, training=training)
        x = self.dense_2(x)
        x = self.add([inputs, x])
        return self.layer_norm(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "hidden_dim": self.hidden_dim,
            "dropout_rate": self.dropout_rate,
        })
        return config


# --------------------------------------------------
# 3. Rebuild the confirmed model architecture
# --------------------------------------------------
inputs = keras.Input(shape=(256,), name="noisy_audio")

x = layers.Reshape((256, 1), name="temporal_reshape")(inputs)
x = layers.Dense(64, activation="relu", name="feature_projection")(x)

x = TemporalMLPBlockRestore(
    hidden_dim=64, dropout_rate=0.1,
    name="temporal_mlp_block_1"
)(x)

x = TemporalMLPBlockRestore(
    hidden_dim=64, dropout_rate=0.1,
    name="temporal_mlp_block_2"
)(x)

x = layers.Dense(32, activation="relu", name="feature_reduction")(x)
x = layers.Dense(1, activation=None, name="output_projection")(x)
outputs = layers.Reshape((256,), name="output_waveform")(x)

restored_model = keras.Model(inputs, outputs, name="T_MLP")

# Build all weights before assigning checkpoint values.
_ = restored_model(np.zeros((1, 256), dtype=np.float32), training=False)

# --------------------------------------------------
# 4. Restore trained weights from the archive
# --------------------------------------------------
def read_weight(h5_file, path):
    return h5_file[path][()]

with zipfile.ZipFile(model_path, "r") as archive:
    weights_bytes = io.BytesIO(archive.read("model.weights.h5"))

    with h5py.File(weights_bytes, "r") as h5:
        # Top-level Dense layers
        restored_model.get_layer("feature_projection").set_weights([
            read_weight(h5, "layers/dense/vars/0"),
            read_weight(h5, "layers/dense/vars/1"),
        ])

        restored_model.get_layer("feature_reduction").set_weights([
            read_weight(h5, "layers/dense_1/vars/0"),
            read_weight(h5, "layers/dense_1/vars/1"),
        ])

        restored_model.get_layer("output_projection").set_weights([
            read_weight(h5, "layers/dense_2/vars/0"),
            read_weight(h5, "layers/dense_2/vars/1"),
        ])

        # Archive group -> corresponding custom block
        block_mapping = {
            "temporal_mlp_block_1": "layers/temporal_mlp_block",
            "temporal_mlp_block_2": "layers/temporal_mlp_block_1",
        }

        for model_layer_name, archive_group in block_mapping.items():
            block = restored_model.get_layer(model_layer_name)

            block.dense_1.set_weights([
                read_weight(h5, f"{archive_group}/dense_1/vars/0"),
                read_weight(h5, f"{archive_group}/dense_1/vars/1"),
            ])

            block.dense_2.set_weights([
                read_weight(h5, f"{archive_group}/dense_2/vars/0"),
                read_weight(h5, f"{archive_group}/dense_2/vars/1"),
            ])

            block.layer_norm.set_weights([
                read_weight(h5, f"{archive_group}/layer_norm/vars/0"),
                read_weight(h5, f"{archive_group}/layer_norm/vars/1"),
            ])

# --------------------------------------------------
# 5. Validate the restored model
# --------------------------------------------------
test_input = np.zeros((2, 256), dtype=np.float32)
test_output = restored_model(test_input, training=False).numpy()

print("RESTORATION CHECK")
print("Input shape:       ", test_input.shape)
print("Output shape:      ", test_output.shape)
print("Parameters:        ", restored_model.count_params())
print("All outputs finite:", np.isfinite(test_output).all())
print("Checkpoint size:   ", round(model_path.stat().st_size / 1024, 2), "KB")

assert test_output.shape == (2, 256)
assert np.isfinite(test_output).all()
assert restored_model.count_params() == 19137

print("\n✅ Model architecture and checkpoint weights restored.")
restored_model.summary()

RESTORATION CHECK
Input shape:        (2, 256)
Output shape:       (2, 256)
Parameters:         19137
All outputs finite: True
Checkpoint size:    292.7 KB

✅ Model architecture and checkpoint weights restored.


Model: "T_MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ noisy_audio (InputLayer)        │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_reshape (Reshape)      │ (None, 256, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_projection (Dense)      │ (None, 256, 64)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_mlp_block_1            │ (None, 256, 64)        │         8,448 │
│ (TemporalMLPBlockRestore)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ temporal_mlp_block_2            │ (None, 256, 64)        │         8,448 │
│ (TemporalMLPBlockRestore)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_reduction (Dense)       │ (None, 256, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_projection (Dense)       │ (None, 256, 1)         │            33 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_waveform (Reshape)       │ (None, 256)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,137 (74.75 KB)

 Trainable params: 19,137 (74.75 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
!pip install tensorflow_model_optimization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 8.0 MB/s eta 0:00:00


In [9]:
import numpy as np
import tensorflow as tf
import tf_keras as tk
import tensorflow_model_optimization as tfmot

print("TensorFlow:", tf.__version__)
print("tf_keras:", tk.__version__)
print("TF-MOT:", tfmot.__version__)

# --------------------------------------------------
# 1. Rebuild the architecture using tf_keras layers
# --------------------------------------------------
def build_tfkeras_tmlp():
    inputs = tk.Input(shape=(256,), name="noisy_audio")
    x = tk.layers.Reshape((256, 1), name="temporal_reshape")(inputs)

    x = tk.layers.Dense(
        64, activation="relu", name="feature_projection"
    )(x)

    block_layer_refs = {}

    for block_num in (1, 2):
        prefix = f"temporal_mlp_block_{block_num}"

        residual = x
        x = tk.layers.Dense(
            64, activation="relu", name=f"{prefix}_dense_1"
        )(x)
        x = tk.layers.Dropout(
            0.1, name=f"{prefix}_dropout"
        )(x)
        x = tk.layers.Dense(
            64, activation=None, name=f"{prefix}_dense_2"
        )(x)
        x = tk.layers.Add(name=f"{prefix}_residual_add")([residual, x])
        x = tk.layers.LayerNormalization(
            name=f"{prefix}_layer_norm"
        )(x)

        block_layer_refs[prefix] = {
            "dense_1": f"{prefix}_dense_1",
            "dense_2": f"{prefix}_dense_2",
            "layer_norm": f"{prefix}_layer_norm",
        }

    x = tk.layers.Dense(
        32, activation="relu", name="feature_reduction"
    )(x)
    x = tk.layers.Dense(
        1, activation=None, name="output_projection"
    )(x)
    outputs = tk.layers.Reshape((256,), name="output_waveform")(x)

    return tk.Model(inputs, outputs, name="T_MLP_tfkeras"), block_layer_refs


qat_base_model, block_layer_refs = build_tfkeras_tmlp()

# --------------------------------------------------
# 2. Copy all trained weights from restored_model
# --------------------------------------------------
qat_base_model.get_layer("feature_projection").set_weights(
    restored_model.get_layer("feature_projection").get_weights()
)
qat_base_model.get_layer("feature_reduction").set_weights(
    restored_model.get_layer("feature_reduction").get_weights()
)
qat_base_model.get_layer("output_projection").set_weights(
    restored_model.get_layer("output_projection").get_weights()
)

for block_num in (1, 2):
    source = restored_model.get_layer(
        f"temporal_mlp_block_{block_num}"
    )
    prefix = f"temporal_mlp_block_{block_num}"

    qat_base_model.get_layer(f"{prefix}_dense_1").set_weights(
        source.dense_1.get_weights()
    )
    qat_base_model.get_layer(f"{prefix}_dense_2").set_weights(
        source.dense_2.get_weights()
    )
    qat_base_model.get_layer(f"{prefix}_layer_norm").set_weights(
        source.layer_norm.get_weights()
    )

# --------------------------------------------------
# 3. Compare outputs on identical deterministic inputs
# --------------------------------------------------
rng = np.random.default_rng(42)
comparison_input = rng.normal(
    size=(8, 256)
).astype(np.float32)

keras3_output = restored_model(
    comparison_input, training=False
).numpy()

tfkeras_output = qat_base_model(
    comparison_input, training=False
).numpy()

max_abs_error = float(
    np.max(np.abs(keras3_output - tfkeras_output))
)
mean_abs_error = float(
    np.mean(np.abs(keras3_output - tfkeras_output))
)

print("\nWEIGHT TRANSFER CHECK")
print("Keras 3 output shape:", keras3_output.shape)
print("tf_keras output shape:", tfkeras_output.shape)
print("Max absolute difference:", max_abs_error)
print("Mean absolute difference:", mean_abs_error)
print("tf_keras parameters:", qat_base_model.count_params())

assert keras3_output.shape == tfkeras_output.shape
assert np.isfinite(tfkeras_output).all()
assert max_abs_error < 1e-4, (
    "The models do not numerically agree. Stop here and share the output."
)

print("\n✅ Weight transfer verified. The original checkpoint is unchanged.")

TensorFlow: 2.21.0
tf_keras: 2.21.0
TF-MOT: 0.8.1

WEIGHT TRANSFER CHECK
Keras 3 output shape: (8, 256)
tf_keras output shape: (8, 256)
Max absolute difference: 8.940696716308594e-08
Mean absolute difference: 1.219086698256433e-08
tf_keras parameters: 19137

✅ Weight transfer verified. The original checkpoint is unchanged.


In [10]:
 import tf_keras as tk
import tensorflow_model_optimization as tfmot
import numpy as np

# --------------------------------------------------
# 1. Annotate Dense layers only
# --------------------------------------------------
def annotate_dense(layer):
    if isinstance(layer, tk.layers.Dense):
        return tfmot.quantization.keras.quantize_annotate_layer(layer)
    return layer

annotated_model = tk.models.clone_model(
    qat_base_model,
    clone_function=annotate_dense
)

# Transfer the trained float weights into the annotated model.
annotated_model.set_weights(qat_base_model.get_weights())

print("Annotated model created.")
print("Annotated model parameters:", annotated_model.count_params())

# --------------------------------------------------
# 2. Apply quantization-aware training wrappers
# --------------------------------------------------
try:
    qat_model = tfmot.quantization.keras.quantize_apply(
        annotated_model
    )
    print("\n✅ QAT wrappers applied successfully.")

except Exception as exc:
    print("\n❌ QAT conversion failed.")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))
    raise

# --------------------------------------------------
# 3. Inspect the resulting model
# --------------------------------------------------
print("\nQAT MODEL CHECK")
print("Input shape:", qat_model.input_shape)
print("Output shape:", qat_model.output_shape)
print("Total parameters:", qat_model.count_params())
print("Trainable parameters:", sum(
    np.prod(w.shape) for w in qat_model.trainable_weights
))
print("Quantization-related layers:")

for layer in qat_model.layers:
    if "quant" in layer.name.lower():
        print(f"  {layer.name}: {layer.__class__.__name__}")

# Confirm the converted model runs.
rng = np.random.default_rng(42)
test_input = rng.normal(size=(4, 256)).astype(np.float32)
test_output = qat_model(test_input, training=False).numpy()

print("\nOutput shape:", test_output.shape)
print("All outputs finite:", np.isfinite(test_output).all())

assert test_output.shape == (4, 256)
assert np.isfinite(test_output).all()

print("\n✅ QAT model passes the structural and inference checks.")

Annotated model created.
Annotated model parameters: 19137

✅ QAT wrappers applied successfully.

QAT MODEL CHECK
Input shape: (None, 256)
Output shape: (None, 256)
Total parameters: 19175
Trainable parameters: 19137
Quantization-related layers:
  quant_temporal_reshape: QuantizeWrapperV2
  quant_feature_projection: QuantizeWrapperV2
  quant_temporal_mlp_block_1_dense_1: QuantizeWrapperV2
  quant_temporal_mlp_block_1_dropout: QuantizeWrapperV2
  quant_temporal_mlp_block_1_dense_2: QuantizeWrapperV2
  quant_temporal_mlp_block_2_dense_1: QuantizeWrapperV2
  quant_temporal_mlp_block_2_dropout: QuantizeWrapperV2
  quant_temporal_mlp_block_2_dense_2: QuantizeWrapperV2
  quant_feature_reduction: QuantizeWrapperV2
  quant_output_projection: QuantizeWrapperV2

Output shape: (4, 256)
All outputs finite: True

✅ QAT model passes the structural and inference checks.


In [11]:
from pathlib import Path
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
data_dir = project_dir / "processed_data"

for split in ("train", "validation", "test"):
    split_dir = data_dir / split
    files = sorted(split_dir.glob("*.npz"))

    print(f"\n{'=' * 55}")
    print(f"{split.upper()}: {len(files)} NPZ files")

    if not files:
        print("WARNING: No NPZ files found.")
        continue

    # Inspect one file without modifying it.
    with np.load(files[0], allow_pickle=True) as data:
        print("Example file:", files[0].name)
        print("Keys:", data.files)

        for key in data.files:
            value = data[key]
            print(
                f"  {key}: shape={value.shape}, "
                f"dtype={value.dtype}"
            )

        # Report basic signal information if the expected keys exist.
        if "noisy_speech" in data.files and "clean_speech" in data.files:
            noisy = np.asarray(data["noisy_speech"], dtype=np.float32)
            clean = np.asarray(data["clean_speech"], dtype=np.float32)

            print("Noisy samples:", noisy.size)
            print("Clean samples:", clean.size)
            print("Matching lengths:", noisy.shape == clean.shape)
            print("Noisy finite:", np.isfinite(noisy).all())
            print("Clean finite:", np.isfinite(clean).all())

# Check whether validation windows already exist in memory.
print("\n" + "=" * 55)
print("EXISTING WINDOW VARIABLES")
for name in ("X_train", "Y_train", "X_val", "Y_val"):
    if name in globals():
        value = globals()[name]
        print(f"{name}: shape={getattr(value, 'shape', 'unknown')}")
    else:
        print(f"{name}: not present")


TRAIN: 800 NPZ files
Example file: sample_000000.npz
Keys: ['noisy_speech', 'clean_speech', 'speech_filename', 'noise_filename', 'target_snr_db']
  noisy_speech: shape=(326240,), dtype=float32
  clean_speech: shape=(326240,), dtype=float32
  speech_filename: shape=(), dtype=<U20
  noise_filename: shape=(), dtype=<U17
  target_snr_db: shape=(), dtype=int64
Noisy samples: 326240
Clean samples: 326240
Matching lengths: True
Noisy finite: True
Clean finite: True

VALIDATION: 100 NPZ files
Example file: sample_000000.npz
Keys: ['noisy_speech', 'clean_speech', 'speech_filename', 'noise_filename', 'target_snr_db']
  noisy_speech: shape=(92640,), dtype=float32
  clean_speech: shape=(92640,), dtype=float32
  speech_filename: shape=(), dtype=<U20
  noise_filename: shape=(), dtype=<U16
  target_snr_db: shape=(), dtype=int64
Noisy samples: 92640
Clean samples: 92640
Matching lengths: True
Noisy finite: True
Clean finite: True

TEST: 100 NPZ files
Example file: sample_000000.npz
Keys: ['noisy_spee

In [12]:
from pathlib import Path
import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

project_dir = Path("/content/drive/MyDrive/T-MLP")
train_dir = project_dir / "processed_data" / "train"

WINDOW_SIZE = 256
MAX_WINDOWS_PER_FILE = 64
SEED = 42

rng = np.random.default_rng(SEED)

# ============================================================
# 2. LOAD TRAINING FILES AND CREATE WINDOWS
# ============================================================

train_files = sorted(train_dir.glob("*.npz"))

if not train_files:
    raise FileNotFoundError(f"No .npz files found in {train_dir}")

X_parts = []
Y_parts = []
files_used = 0
windows_created = 0

for file_path in train_files:
    with np.load(file_path, allow_pickle=True) as data:
        noisy = np.asarray(data["noisy_speech"], dtype=np.float32)
        clean = np.asarray(data["clean_speech"], dtype=np.float32)

    if noisy.shape != clean.shape:
        raise ValueError(f"Length mismatch in {file_path.name}")

    if not np.isfinite(noisy).all() or not np.isfinite(clean).all():
        raise ValueError(f"Non-finite samples in {file_path.name}")

    # Number of complete, non-overlapping windows.
    available = len(noisy) // WINDOW_SIZE

    if available == 0:
        continue

    count = min(MAX_WINDOWS_PER_FILE, available)

    # Randomly sample windows without replacement.
    indices = rng.choice(available, size=count, replace=False)
    starts = indices * WINDOW_SIZE

    noisy_windows = np.stack([
        noisy[start:start + WINDOW_SIZE]
        for start in starts
    ])

    clean_windows = np.stack([
        clean[start:start + WINDOW_SIZE]
        for start in starts
    ])

    X_parts.append(noisy_windows)
    Y_parts.append(clean_windows)

    files_used += 1
    windows_created += count

if not X_parts:
    raise RuntimeError("No training windows were created.")

X_train = np.concatenate(X_parts).astype(np.float32, copy=False)
Y_train = np.concatenate(Y_parts).astype(np.float32, copy=False)

# Shuffle input/target pairs together.
permutation = rng.permutation(len(X_train))
X_train = X_train[permutation]
Y_train = Y_train[permutation]

# ============================================================
# 3. TRAINING DATA SUMMARY
# ============================================================

print("=" * 60)
print("TRAINING WINDOW SUMMARY")
print("=" * 60)
print("Files available:", len(train_files))
print("Files used:", files_used)
print("Windows created:", windows_created)
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_train dtype:", X_train.dtype)
print("Y_train dtype:", Y_train.dtype)
print("X_train finite:", np.isfinite(X_train).all())
print("Y_train finite:", np.isfinite(Y_train).all())

print("\nTRAIN SIGNAL RANGES")
print("Noisy min/max:", float(X_train.min()), float(X_train.max()))
print("Clean min/max:", float(Y_train.min()), float(Y_train.max()))

assert X_train.shape == Y_train.shape
assert X_train.ndim == 2 and X_train.shape[1] == WINDOW_SIZE
assert np.isfinite(X_train).all() and np.isfinite(Y_train).all()

# ============================================================
# 4. LOAD MATCHED VALIDATION WINDOWS
# ============================================================

# These were created earlier for the QAT comparison.
# Do not silently substitute a different validation set.
if "X_val_matched" not in globals() or "Y_val_matched" not in globals():
    raise RuntimeError(
        "Matched validation arrays are missing from this Colab session. "
        "Recreate X_val_matched and Y_val_matched from the existing "
        "processed_data/validation files before continuing."
    )

X_val = np.asarray(X_val_matched, dtype=np.float32)
Y_val = np.asarray(Y_val_matched, dtype=np.float32)

assert X_val.ndim == 2 and X_val.shape[1] == WINDOW_SIZE
assert X_val.shape == Y_val.shape
assert np.isfinite(X_val).all() and np.isfinite(Y_val).all()

print("\n" + "=" * 60)
print("MATCHED VALIDATION WINDOWS")
print("=" * 60)
print("X_val shape:", X_val.shape)
print("Y_val shape:", Y_val.shape)

# ============================================================
# 5. VERIFY FLOAT MODEL IS AVAILABLE
# ============================================================

if "restored_model" not in globals():
    raise RuntimeError(
        "The restored float T-MLP is not available in this session. "
        "Restore the float model before running the baseline evaluation."
    )

# ============================================================
# 6. FLOAT MODEL BASELINE EVALUATION
# ============================================================

float_predictions = restored_model.predict(
    X_val,
    batch_size=256,
    verbose=0
)

def calculate_metrics(reference, prediction):
    error = reference - prediction
    return {
        "MSE": float(np.mean(np.square(error))),
        "MAE": float(np.mean(np.abs(error))),
    }

noisy_metrics = calculate_metrics(Y_val, X_val)
float_metrics = calculate_metrics(Y_val, float_predictions)

print("\n" + "=" * 60)
print("FLOAT MODEL VALIDATION BASELINE")
print("=" * 60)
print(f"Noisy input MSE: {noisy_metrics['MSE']:.8f}")
print(f"Float model MSE: {float_metrics['MSE']:.8f}")
print(f"Noisy input MAE: {noisy_metrics['MAE']:.8f}")
print(f"Float model MAE: {float_metrics['MAE']:.8f}")

if noisy_metrics["MSE"] > 0:
    improvement = (
        (noisy_metrics["MSE"] - float_metrics["MSE"])
        / noisy_metrics["MSE"]
    ) * 100

    print(f"MSE improvement over noisy input: {improvement:.2f}%")

assert np.isfinite(float_predictions).all()

print("\nAll checks passed.")
print("Training windows prepared; no model was trained.")
print("Saved checkpoints were not modified.")
print("Test data remains untouched.")

TRAINING WINDOW SUMMARY
Files available: 800
Files used: 800
Windows created: 51200
X_train shape: (51200, 256)
Y_train shape: (51200, 256)
X_train dtype: float32
Y_train dtype: float32
X_train finite: True
Y_train finite: True

TRAIN SIGNAL RANGES
Noisy min/max: -9.50479793548584 8.960302352905273
Clean min/max: -0.96575927734375 0.988250732421875


RuntimeError: Matched validation arrays are missing from this Colab session. Recreate X_val_matched and Y_val_matched from the existing processed_data/validation files before continuing.

In [13]:
from pathlib import Path
import numpy as np

# ============================================================
# 1. CONFIGURATION
# ============================================================

project_dir = Path("/content/drive/MyDrive/T-MLP")
val_dir = project_dir / "processed_data" / "validation"

WINDOW_SIZE = 256
MAX_WINDOWS_PER_FILE = 64
SEED = 2026

rng = np.random.default_rng(SEED)

# ============================================================
# 2. RECREATE MATCHED VALIDATION WINDOWS
# ============================================================

val_files = sorted(val_dir.glob("*.npz"))

if not val_files:
    raise FileNotFoundError(f"No validation .npz files found in {val_dir}")

X_val_parts = []
Y_val_parts = []
files_used = 0

for file_path in val_files:
    with np.load(file_path, allow_pickle=True) as data:
        noisy = np.asarray(data["noisy_speech"], dtype=np.float32)
        clean = np.asarray(data["clean_speech"], dtype=np.float32)

    if noisy.shape != clean.shape:
        raise ValueError(f"Signal length mismatch: {file_path.name}")

    if not np.isfinite(noisy).all() or not np.isfinite(clean).all():
        raise ValueError(f"Non-finite values in {file_path.name}")

    available = len(noisy) // WINDOW_SIZE
    if available == 0:
        continue

    count = min(MAX_WINDOWS_PER_FILE, available)
    indices = rng.choice(available, size=count, replace=False)
    starts = indices * WINDOW_SIZE

    X_val_parts.append(np.stack([
        noisy[start:start + WINDOW_SIZE] for start in starts
    ]))

    Y_val_parts.append(np.stack([
        clean[start:start + WINDOW_SIZE] for start in starts
    ]))

    files_used += 1

if not X_val_parts:
    raise RuntimeError("No validation windows could be created.")

X_val_matched = np.concatenate(X_val_parts).astype(np.float32)
Y_val_matched = np.concatenate(Y_val_parts).astype(np.float32)

# Keep the variable names used in evaluation.
X_val = X_val_matched
Y_val = Y_val_matched

print("=" * 60)
print("VALIDATION DATA")
print("=" * 60)
print("Validation files found:", len(val_files))
print("Validation files used:", files_used)
print("X_val shape:", X_val.shape)
print("Y_val shape:", Y_val.shape)
print("Noisy range:", float(X_val.min()), float(X_val.max()))
print("Clean range:", float(Y_val.min()), float(Y_val.max()))

assert X_val.shape == Y_val.shape
assert X_val.ndim == 2 and X_val.shape[1] == WINDOW_SIZE
assert np.isfinite(X_val).all() and np.isfinite(Y_val).all()

# ============================================================
# 3. CHECK MODEL AVAILABILITY
# ============================================================

if "restored_model" not in globals():
    raise RuntimeError(
        "restored_model is missing. Restore your original float T-MLP "
        "checkpoint using the restoration code from the earlier notebook."
    )

# Use the fine-tuned QAT model and best saved weights if available.
qat_weights_path = (
    project_dir / "saved_models" / "TMLP_QAT_final_best.weights.h5"
)

if "qat_model" not in globals():
    raise RuntimeError(
        "qat_model is missing. Rebuild the QAT model architecture using "
        "your earlier TF-MOT quantization setup before loading its weights."
    )

if not qat_weights_path.exists():
    raise FileNotFoundError(f"QAT weights not found: {qat_weights_path}")

qat_model.load_weights(str(qat_weights_path))

# ============================================================
# 4. EVALUATE FLOAT AND QAT MODELS
# ============================================================

def metrics(reference, prediction):
    error = reference - prediction
    return (
        float(np.mean(np.square(error))),
        float(np.mean(np.abs(error)))
    )

float_pred = restored_model.predict(X_val, batch_size=256, verbose=0)
qat_pred = qat_model.predict(X_val, batch_size=256, verbose=0)

noisy_mse, noisy_mae = metrics(Y_val, X_val)
float_mse, float_mae = metrics(Y_val, float_pred)
qat_mse, qat_mae = metrics(Y_val, qat_pred)

print("\n" + "=" * 72)
print("VALIDATION RESULTS")
print("=" * 72)
print(f"{'Model':<20} {'MSE':>15} {'MAE':>15}")
print("-" * 52)
print(f"{'Noisy input':<20} {noisy_mse:>15.8f} {noisy_mae:>15.8f}")
print(f"{'Float T-MLP':<20} {float_mse:>15.8f} {float_mae:>15.8f}")
print(f"{'QAT T-MLP':<20} {qat_mse:>15.8f} {qat_mae:>15.8f}")

mse_change = ((qat_mse - float_mse) / float_mse) * 100
mae_change = ((qat_mae - float_mae) / float_mae) * 100

print("\nQAT versus Float T-MLP:")
print(f"MSE change: {mse_change:+.2f}%")
print(f"MAE change: {mae_change:+.2f}%")
print(
    "Mean absolute prediction difference:",
    f"{np.mean(np.abs(qat_pred - float_pred)):.8f}"
)

print("\nEvaluation completed. No training performed.")

VALIDATION DATA
Validation files found: 100
Validation files used: 100
X_val shape: (6400, 256)
Y_val shape: (6400, 256)
Noisy range: -1.5779329538345337 1.542458415031433
Clean range: -0.708770751953125 0.808929443359375

VALIDATION RESULTS
Model                            MSE             MAE
----------------------------------------------------
Noisy input               0.00256030      0.02096159
Float T-MLP               0.00123290      0.02033874
QAT T-MLP                 0.00123569      0.02030816

QAT versus Float T-MLP:
MSE change: +0.23%
MAE change: -0.15%
Mean absolute prediction difference: 0.00171806

Evaluation completed. No training performed.


In [14]:
from pathlib import Path
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
train_dir = project_dir / "processed_data" / "train"

WINDOW_SIZE = 256
MAX_WINDOWS_PER_FILE = 64
SEED = 42

rng = np.random.default_rng(SEED)
train_files = sorted(train_dir.glob("*.npz"))

X_parts = []
Y_parts = []
files_used = 0
windows_created = 0

for file_path in train_files:
    with np.load(file_path, allow_pickle=True) as data:
        noisy = np.asarray(data["noisy_speech"], dtype=np.float32)
        clean = np.asarray(data["clean_speech"], dtype=np.float32)

    if noisy.shape != clean.shape:
        raise ValueError(f"Length mismatch in {file_path.name}")

    if not np.isfinite(noisy).all() or not np.isfinite(clean).all():
        raise ValueError(f"Non-finite samples in {file_path.name}")

    # Number of complete, non-overlapping windows available.
    available = len(noisy) // WINDOW_SIZE
    if available == 0:
        continue

    count = min(MAX_WINDOWS_PER_FILE, available)

    # Randomly choose window indices without replacement.
    indices = rng.choice(available, size=count, replace=False)
    starts = indices * WINDOW_SIZE

    noisy_windows = np.stack([
        noisy[start:start + WINDOW_SIZE] for start in starts
    ])
    clean_windows = np.stack([
        clean[start:start + WINDOW_SIZE] for start in starts
    ])

    X_parts.append(noisy_windows)
    Y_parts.append(clean_windows)

    files_used += 1
    windows_created += count

if not X_parts:
    raise RuntimeError("No training windows were created.")

X_train = np.concatenate(X_parts).astype(np.float32, copy=False)
Y_train = np.concatenate(Y_parts).astype(np.float32, copy=False)

# Shuffle paired noisy/clean windows together.
permutation = rng.permutation(len(X_train))
X_train = X_train[permutation]
Y_train = Y_train[permutation]

print("TRAINING WINDOW SUMMARY")
print("Files available:", len(train_files))
print("Files used:", files_used)
print("Windows created:", windows_created)
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_train dtype:", X_train.dtype)
print("Y_train dtype:", Y_train.dtype)
print("X_train finite:", np.isfinite(X_train).all())
print("Y_train finite:", np.isfinite(Y_train).all())

print("\nTRAIN SIGNAL RANGES")
print("Noisy min/max:", float(X_train.min()), float(X_train.max()))
print("Clean min/max:", float(Y_train.min()), float(Y_train.max()))

# Confirm the existing validation arrays are usable and unchanged.
assert "X_val" in globals() and "Y_val" in globals()
X_val = np.asarray(X_val, dtype=np.float32)
Y_val = np.asarray(Y_val, dtype=np.float32)

assert X_val.ndim == 2 and X_val.shape[1] == WINDOW_SIZE
assert X_val.shape == Y_val.shape
assert np.isfinite(X_val).all() and np.isfinite(Y_val).all()

print("\nEXISTING VALIDATION WINDOWS")
print("X_val shape:", X_val.shape)
print("Y_val shape:", Y_val.shape)

# Establish float-model baseline metrics on the same validation windows.
float_predictions = restored_model.predict(
    X_val, batch_size=256, verbose=0
)

noisy_mse = float(np.mean(np.square(X_val - Y_val)))
float_mse = float(np.mean(np.square(float_predictions - Y_val)))
noisy_mae = float(np.mean(np.abs(X_val - Y_val)))
float_mae = float(np.mean(np.abs(float_predictions - Y_val)))

print("\nFLOAT MODEL VALIDATION BASELINE")
print(f"Noisy input MSE: {noisy_mse:.8f}")
print(f"Float model MSE: {float_mse:.8f}")
print(f"Noisy input MAE: {noisy_mae:.8f}")
print(f"Float model MAE: {float_mae:.8f}")

if noisy_mse > 0:
    print(
        "MSE improvement over noisy input:",
        f"{100 * (noisy_mse - float_mse) / noisy_mse:.2f}%"
    )

assert len(X_train) > 0
assert np.isfinite(float_predictions).all()

print("\n✅ Training windows prepared; test data remains untouched.")

TRAINING WINDOW SUMMARY
Files available: 800
Files used: 800
Windows created: 51200
X_train shape: (51200, 256)
Y_train shape: (51200, 256)
X_train dtype: float32
Y_train dtype: float32
X_train finite: True
Y_train finite: True

TRAIN SIGNAL RANGES
Noisy min/max: -9.50479793548584 8.960302352905273
Clean min/max: -0.96575927734375 0.988250732421875

EXISTING VALIDATION WINDOWS
X_val shape: (6400, 256)
Y_val shape: (6400, 256)

FLOAT MODEL VALIDATION BASELINE
Noisy input MSE: 0.00256030
Float model MSE: 0.00123290
Noisy input MAE: 0.02096159
Float model MAE: 0.02033874
MSE improvement over noisy input: 51.85%

✅ Training windows prepared; test data remains untouched.


In [15]:
from pathlib import Path
import numpy as np
import tensorflow as tf
import tf_keras as tk

# --------------------------------------------------
# 1. Set up separate checkpoint paths
# --------------------------------------------------
project_dir = Path("/content/drive/MyDrive/T-MLP")
model_dir = project_dir / "saved_models"
model_dir.mkdir(parents=True, exist_ok=True)

initial_weights_path = model_dir / "TMLP_QAT_initial.weights.h5"
best_weights_path = model_dir / "TMLP_QAT_best.weights.h5"

# Never overwrite an existing QAT checkpoint accidentally.
if initial_weights_path.exists() or best_weights_path.exists():
    raise FileExistsError(
        "A QAT checkpoint already exists. Inspect it before continuing:\n"
        f"{initial_weights_path}\n{best_weights_path}"
    )

# --------------------------------------------------
# 2. Evaluate QAT BEFORE fine-tuning
# --------------------------------------------------
qat_before = qat_model.predict(
    X_val, batch_size=256, verbose=0
)

qat_before_mse = float(np.mean(np.square(qat_before - Y_val)))
qat_before_mae = float(np.mean(np.abs(qat_before - Y_val)))

print("PRE-FINETUNING QAT BASELINE")
print(f"QAT MSE: {qat_before_mse:.8f}")
print(f"QAT MAE: {qat_before_mae:.8f}")
print(
    "MSE difference vs float model:",
    f"{qat_before_mse - float_mse:+.8f}"
)

# Preserve the exact initial QAT weights.
qat_model.save_weights(str(initial_weights_path))
print("\nInitial QAT weights saved:", initial_weights_path)

# --------------------------------------------------
# 3. Compile with a conservative learning rate
# --------------------------------------------------
qat_model.compile(
    optimizer=tk.optimizers.Adam(learning_rate=1e-5),
    loss="mse",
    metrics=["mae"],
)

callbacks = [
    tk.callbacks.ModelCheckpoint(
        filepath=str(best_weights_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tk.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    tk.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

# --------------------------------------------------
# 4. Fine-tune
# --------------------------------------------------
history = qat_model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=10,
    batch_size=256,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)

# --------------------------------------------------
# 5. Evaluate the restored best QAT weights
# --------------------------------------------------
qat_after = qat_model.predict(
    X_val, batch_size=256, verbose=0
)

qat_after_mse = float(np.mean(np.square(qat_after - Y_val)))
qat_after_mae = float(np.mean(np.abs(qat_after - Y_val)))

print("\n" + "=" * 55)
print("QAT FINE-TUNING SUMMARY")
print("=" * 55)
print("Epochs completed:", len(history.history["loss"]))
print(f"Float baseline MSE:       {float_mse:.8f}")
print(f"QAT before fine-tuning:   {qat_before_mse:.8f}")
print(f"QAT after fine-tuning:    {qat_after_mse:.8f}")
print(f"Float baseline MAE:       {float_mae:.8f}")
print(f"QAT before MAE:           {qat_before_mae:.8f}")
print(f"QAT after MAE:            {qat_after_mae:.8f}")
print("Best weights exist:", best_weights_path.exists())

if qat_before_mse > 0:
    print(
        "QAT MSE change after fine-tuning:",
        f"{100 * (qat_before_mse - qat_after_mse) / qat_before_mse:.2f}%"
    )

if qat_after_mse < float_mse:
    print("QAT validation MSE is lower than the float baseline.")
else:
    print("QAT has not surpassed the float baseline; inspect the metrics.")

assert best_weights_path.exists(), (
    "No best checkpoint was saved. Check the training logs."
)
assert np.isfinite(qat_after).all()

print("\n✅ Fine-tuning run completed and best weights are preserved.")

FileExistsError: A QAT checkpoint already exists. Inspect it before continuing:
/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_initial.weights.h5
/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_best.weights.h5

In [16]:
import numpy as np

# --------------------------------------------------
# 1. Signal distribution comparison
# --------------------------------------------------
def signal_stats(name, x, y):
    noise = x - y

    print(f"\n{name}")
    print("-" * 48)
    print(f"Noisy mean:          {np.mean(x): .6f}")
    print(f"Noisy std:           {np.std(x): .6f}")
    print(f"Noisy RMS:           {np.sqrt(np.mean(x**2)): .6f}")
    print(f"Noisy |x| p99:       {np.percentile(np.abs(x), 99): .6f}")
    print(f"Clean mean:          {np.mean(y): .6f}")
    print(f"Clean std:           {np.std(y): .6f}")
    print(f"Clean RMS:           {np.sqrt(np.mean(y**2)): .6f}")
    print(f"Noise RMS:           {np.sqrt(np.mean(noise**2)): .6f}")
    print(f"Noisy/clean shape:   {x.shape} / {y.shape}")

signal_stats("TRAIN WINDOWS", X_train, Y_train)
signal_stats("VALIDATION WINDOWS", X_val, Y_val)

# --------------------------------------------------
# 2. Compare float and best QAT on identical windows
# --------------------------------------------------
float_val_pred = restored_model.predict(
    X_val, batch_size=256, verbose=0
)
qat_val_pred = qat_model.predict(
    X_val, batch_size=256, verbose=0
)

def metrics(name, prediction, target):
    error = prediction - target
    print(
        f"{name:<18} "
        f"MSE={np.mean(error**2):.8f}  "
        f"MAE={np.mean(np.abs(error)):.8f}  "
        f"Pred RMS={np.sqrt(np.mean(prediction**2)):.6f}  "
        f"Finite={np.isfinite(prediction).all()}"
    )

print("\nVALIDATION PERFORMANCE")
print("-" * 75)
metrics("Noisy input", X_val, Y_val)
metrics("Float model", float_val_pred, Y_val)
metrics("Best QAT model", qat_val_pred, Y_val)

# --------------------------------------------------
# 3. Quantify the prediction difference
# --------------------------------------------------
difference = qat_val_pred - float_val_pred

print("\nQAT vs FLOAT PREDICTIONS")
print("-" * 48)
print("Mean absolute difference:", float(np.mean(np.abs(difference))))
print("Maximum absolute difference:", float(np.max(np.abs(difference))))
print("Difference RMS:", float(np.sqrt(np.mean(difference**2))))

# --------------------------------------------------
# 4. Compare per-window errors
# --------------------------------------------------
float_window_mse = np.mean((float_val_pred - Y_val)**2, axis=1)
qat_window_mse = np.mean((qat_val_pred - Y_val)**2, axis=1)

print("\nPER-WINDOW VALIDATION ERRORS")
print("-" * 48)
print("Float median MSE:", float(np.median(float_window_mse)))
print("QAT median MSE:", float(np.median(qat_window_mse)))
print("Float 95th percentile:", float(np.percentile(float_window_mse, 95)))
print("QAT 95th percentile:", float(np.percentile(qat_window_mse, 95)))
print(
    "Windows where QAT beats float:",
    f"{100*np.mean(qat_window_mse < float_window_mse):.2f}%"
)

# --------------------------------------------------
# 5. Check that the original float checkpoint is untouched
# --------------------------------------------------
from pathlib import Path

float_checkpoint = Path(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras"
)
qat_checkpoint = Path(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_best.weights.h5"
)

print("\nCHECKPOINTS")
print("Original float checkpoint exists:", float_checkpoint.exists())
print("Best QAT weights exist:", qat_checkpoint.exists())
if float_checkpoint.exists():
    print("Float checkpoint size (KB):", round(float_checkpoint.stat().st_size / 1024, 2))
if qat_checkpoint.exists():
    print("QAT weights size (KB):", round(qat_checkpoint.stat().st_size / 1024, 2))


TRAIN WINDOWS
------------------------------------------------
Noisy mean:          -0.000210
Noisy std:            0.088966
Noisy RMS:            0.088966
Noisy |x| p99:        0.338896
Clean mean:          -0.000245
Clean std:            0.064711
Clean RMS:            0.064711
Noise RMS:            0.061061
Noisy/clean shape:   (51200, 256) / (51200, 256)

VALIDATION WINDOWS
------------------------------------------------
Noisy mean:           0.000353
Noisy std:            0.073548
Noisy RMS:            0.073549
Noisy |x| p99:        0.279595
Clean mean:           0.000588
Clean std:            0.053212
Clean RMS:            0.053215
Noise RMS:            0.050599
Noisy/clean shape:   (6400, 256) / (6400, 256)

VALIDATION PERFORMANCE
---------------------------------------------------------------------------
Noisy input        MSE=0.00256030  MAE=0.02096159  Pred RMS=0.073549  Finite=True
Float model        MSE=0.00123290  MAE=0.02033874  Pred RMS=0.039728  Finite=True
Best QAT mo

In [17]:
from pathlib import Path
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
validation_dir = project_dir / "processed_data" / "validation"

WINDOW_SIZE = 256
MAX_WINDOWS_PER_FILE = 64

rng_val = np.random.default_rng(2026)
validation_files = sorted(validation_dir.glob("*.npz"))

X_val_matched_parts = []
Y_val_matched_parts = []

for file_path in validation_files:
    with np.load(file_path, allow_pickle=True) as data:
        noisy = np.asarray(data["noisy_speech"], dtype=np.float32)
        clean = np.asarray(data["clean_speech"], dtype=np.float32)

    if noisy.shape != clean.shape:
        raise ValueError(f"Length mismatch: {file_path.name}")

    if not np.isfinite(noisy).all() or not np.isfinite(clean).all():
        raise ValueError(f"Non-finite audio: {file_path.name}")

    available = len(noisy) // WINDOW_SIZE
    if available == 0:
        continue

    count = min(MAX_WINDOWS_PER_FILE, available)
    indices = rng_val.choice(available, size=count, replace=False)
    starts = indices * WINDOW_SIZE

    X_val_matched_parts.append(np.stack([
        noisy[s:s + WINDOW_SIZE] for s in starts
    ]))
    Y_val_matched_parts.append(np.stack([
        clean[s:s + WINDOW_SIZE] for s in starts
    ]))

X_val_matched = np.concatenate(X_val_matched_parts).astype(np.float32)
Y_val_matched = np.concatenate(Y_val_matched_parts).astype(np.float32)

# Evaluate both models on exactly the same new windows.
float_matched_pred = restored_model.predict(
    X_val_matched, batch_size=256, verbose=0
)
qat_matched_pred = qat_model.predict(
    X_val_matched, batch_size=256, verbose=0
)

def report_metrics(name, prediction, target):
    mse = float(np.mean((prediction - target) ** 2))
    mae = float(np.mean(np.abs(prediction - target)))
    print(f"{name:<20} MSE={mse:.8f}  MAE={mae:.8f}")

print("MATCHED VALIDATION SET")
print("Files:", len(validation_files))
print("Windows:", len(X_val_matched))
print("Shape:", X_val_matched.shape)

print("\nSIGNAL DISTRIBUTIONS")
print("Training noisy RMS:", float(np.sqrt(np.mean(X_train ** 2))))
print("Matched validation noisy RMS:",
      float(np.sqrt(np.mean(X_val_matched ** 2))))
print("Training clean RMS:", float(np.sqrt(np.mean(Y_train ** 2))))
print("Matched validation clean RMS:",
      float(np.sqrt(np.mean(Y_val_matched ** 2))))

print("\nPERFORMANCE ON MATCHED VALIDATION")
report_metrics("Noisy input", X_val_matched, Y_val_matched)
report_metrics("Float T-MLP", float_matched_pred, Y_val_matched)
report_metrics("Best QAT", qat_matched_pred, Y_val_matched)

print("\nOriginal validation arrays preserved:", X_val.shape == (2048, 256))
print("Test split untouched.")

MATCHED VALIDATION SET
Files: 100
Windows: 6400
Shape: (6400, 256)

SIGNAL DISTRIBUTIONS
Training noisy RMS: 0.0889659970998764
Matched validation noisy RMS: 0.07354902476072311
Training clean RMS: 0.0647110715508461
Matched validation clean RMS: 0.05321499705314636

PERFORMANCE ON MATCHED VALIDATION
Noisy input          MSE=0.00256030  MAE=0.02096159
Float T-MLP          MSE=0.00123290  MAE=0.02033874
Best QAT             MSE=0.00123569  MAE=0.02030816

Original validation arrays preserved: False
Test split untouched.


In [18]:
from pathlib import Path
from collections import Counter
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
data_dir = project_dir / "processed_data"

for split in ("train", "validation", "test"):
    files = sorted((data_dir / split).glob("*.npz"))

    snrs = []
    noisy_rms = []
    clean_rms = []
    noise_rms = []
    speech_names = set()
    noise_names = set()

    for path in files:
        with np.load(path, allow_pickle=True) as data:
            noisy = np.asarray(data["noisy_speech"], dtype=np.float64)
            clean = np.asarray(data["clean_speech"], dtype=np.float64)
            snr = int(data["target_snr_db"])

            snrs.append(snr)
            noisy_rms.append(np.sqrt(np.mean(noisy ** 2)))
            clean_rms.append(np.sqrt(np.mean(clean ** 2)))
            noise_rms.append(np.sqrt(np.mean((noisy - clean) ** 2)))

            speech_names.add(str(data["speech_filename"].item()))
            noise_names.add(str(data["noise_filename"].item()))

    print(f"\n{'=' * 55}")
    print(f"{split.upper()} — {len(files)} files")
    print("SNR counts:", dict(sorted(Counter(snrs).items())))
    print("Unique speech files:", len(speech_names))
    print("Unique noise files:", len(noise_names))
    print(f"Mean noisy RMS: {np.mean(noisy_rms):.6f}")
    print(f"Mean clean RMS: {np.mean(clean_rms):.6f}")
    print(f"Mean noise RMS: {np.mean(noise_rms):.6f}")

    for snr_value in sorted(set(snrs)):
        indices = [i for i, value in enumerate(snrs) if value == snr_value]
        print(
            f"  SNR {snr_value:>3} dB | "
            f"files={len(indices):>3} | "
            f"noisy RMS={np.mean(np.array(noisy_rms)[indices]):.6f} | "
            f"clean RMS={np.mean(np.array(clean_rms)[indices]):.6f}"
        )


TRAIN — 800 files
SNR counts: {-5: 160, 0: 160, 5: 160, 10: 160, 15: 160}
Unique speech files: 743
Unique noise files: 661
Mean noisy RMS: 0.079355
Mean clean RMS: 0.059501
Mean noise RMS: 0.045739
  SNR  -5 dB | files=160 | noisy RMS=0.121503 | clean RMS=0.059559
  SNR   0 dB | files=160 | noisy RMS=0.082940 | clean RMS=0.058648
  SNR   5 dB | files=160 | noisy RMS=0.072247 | clean RMS=0.062960
  SNR  10 dB | files=160 | noisy RMS=0.060945 | clean RMS=0.058109
  SNR  15 dB | files=160 | noisy RMS=0.059139 | clean RMS=0.058230

VALIDATION — 100 files
SNR counts: {-5: 20, 0: 20, 5: 20, 10: 20, 15: 20}
Unique speech files: 86
Unique noise files: 98
Mean noisy RMS: 0.068838
Mean clean RMS: 0.051339
Mean noise RMS: 0.039926
  SNR  -5 dB | files= 20 | noisy RMS=0.110753 | clean RMS=0.054291
  SNR   0 dB | files= 20 | noisy RMS=0.068225 | clean RMS=0.048254
  SNR   5 dB | files= 20 | noisy RMS=0.059378 | clean RMS=0.051764
  SNR  10 dB | files= 20 | noisy RMS=0.056985 | clean RMS=0.054304
 

In [19]:
from pathlib import Path
from collections import defaultdict
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
data_dir = project_dir / "processed_data"

split_records = {}
all_speech = {}
all_noise = {}

print("ACTUAL SNR AUDIT")

for split in ("train", "validation", "test"):
    records = []
    speech_names = set()
    noise_names = set()

    for path in sorted((data_dir / split).glob("*.npz")):
        with np.load(path, allow_pickle=True) as data:
            clean = np.asarray(data["clean_speech"], dtype=np.float64)
            noisy = np.asarray(data["noisy_speech"], dtype=np.float64)
            target_snr = int(data["target_snr_db"])

            speech_name = str(data["speech_filename"].item())
            noise_name = str(data["noise_filename"].item())

            # The residual is the noise component present in the mixture.
            noise_component = noisy - clean

            clean_rms = np.sqrt(np.mean(clean ** 2))
            noise_rms = np.sqrt(np.mean(noise_component ** 2))

            if clean_rms > 0 and noise_rms > 0:
                actual_snr = 20 * np.log10(clean_rms / noise_rms)
            else:
                actual_snr = np.nan

            records.append({
                "target_snr": target_snr,
                "actual_snr": actual_snr,
            })

            speech_names.add(speech_name)
            noise_names.add(noise_name)

    split_records[split] = records
    all_speech[split] = speech_names
    all_noise[split] = noise_names

    print(f"\n{split.upper()}")

    for target in (-5, 0, 5, 10, 15):
        values = np.array([
            r["actual_snr"]
            for r in records
            if r["target_snr"] == target
        ])
        values = values[np.isfinite(values)]

        if len(values):
            errors = values - target
            print(
                f"Target {target:>3} dB | "
                f"actual mean={np.mean(values):7.2f} dB | "
                f"median error={np.median(errors):7.2f} dB | "
                f"mean absolute error={np.mean(np.abs(errors)):6.2f} dB"
            )

print("\nSPLIT OVERLAP AUDIT")

for a, b in (
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
):
    speech_overlap = all_speech[a] & all_speech[b]
    noise_overlap = all_noise[a] & all_noise[b]

    print(f"\n{a} vs {b}")
    print("Overlapping speech filenames:", len(speech_overlap))
    print("Overlapping noise filenames:", len(noise_overlap))

    if speech_overlap:
        print("Speech examples:", list(sorted(speech_overlap))[:5])


ACTUAL SNR AUDIT

TRAIN
Target  -5 dB | actual mean=  -5.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target   0 dB | actual mean=   0.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target   5 dB | actual mean=   5.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target  10 dB | actual mean=  10.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target  15 dB | actual mean=  15.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB

VALIDATION
Target  -5 dB | actual mean=  -5.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target   0 dB | actual mean=   0.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target   5 dB | actual mean=   5.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target  10 dB | actual mean=  10.00 dB | median error=   0.00 dB | mean absolute error=  0.00 dB
Target  15 dB | actual mean=  15.00 dB | median error=  -0.00 dB | mean absolute error=  0.

In [20]:
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/T-MLP")

for folder in [
    project_dir / "data_pipeline",
    project_dir / "data" / "speech",
    project_dir / "data" / "noise",
]:
    print(f"{folder}: {'EXISTS' if folder.exists() else 'MISSING'}")

print(
    "Existing dataset:",
    (project_dir / "processed_data").exists()
)
print(
    "New dataset already exists:",
    (project_dir / "processed_data_source_disjoint").exists()
)

/content/drive/MyDrive/T-MLP/data_pipeline: EXISTS
/content/drive/MyDrive/T-MLP/data/speech: MISSING
/content/drive/MyDrive/T-MLP/data/noise: MISSING
Existing dataset: True
New dataset already exists: False


In [21]:
import numpy as np
import tensorflow as tf
from pathlib import Path

# Confirm that the existing QAT setup is available.
assert "qat_model" in globals(), "QAT model not found. Run the QAT setup cell first."
assert "X_train" in globals() and "Y_train" in globals()
assert "X_val_matched" in globals() and "Y_val_matched" in globals()

project_dir = Path("/content/drive/MyDrive/T-MLP")
model_dir = project_dir / "saved_models"
model_dir.mkdir(parents=True, exist_ok=True)

# Keep the existing best weights safe.
best_path = model_dir / "TMLP_QAT_best.weights.h5"
backup_path = model_dir / "TMLP_QAT_best_before_final.weights.h5"

if best_path.exists() and not backup_path.exists():
    import shutil
    shutil.copy2(best_path, backup_path)
    print("Backed up existing best QAT weights.")

# Use the existing training windows and matched validation windows.
Xtr = np.asarray(X_train, dtype=np.float32)
Ytr = np.asarray(Y_train, dtype=np.float32)
Xv = np.asarray(X_val_matched, dtype=np.float32)
Yv = np.asarray(Y_val_matched, dtype=np.float32)

assert Xtr.shape == Ytr.shape
assert Xv.shape == Yv.shape
assert Xtr.shape[1:] == (256,)
assert np.isfinite(Xtr).all() and np.isfinite(Ytr).all()
assert np.isfinite(Xv).all() and np.isfinite(Yv).all()

# Restore the previous best QAT checkpoint before continuing.
if best_path.exists():
    qat_model.load_weights(str(best_path))
    print("Loaded existing best QAT weights.")

qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="mse",
    metrics=["mae"],
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(model_dir / "TMLP_QAT_final_best.weights.h5"),
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

history = qat_model.fit(
    Xtr, Ytr,
    validation_data=(Xv, Yv),
    epochs=10,
    batch_size=128,
    callbacks=callbacks,
    shuffle=True,
    verbose=1,
)

# Save the final best model weights to a new checkpoint.
qat_model.save_weights(
    str(model_dir / "TMLP_QAT_final_best.weights.h5")
)

print("\nQAT fine-tuning complete.")
print("Checkpoint:", model_dir / "TMLP_QAT_final_best.weights.h5")
print("Best validation MSE:", min(history.history["val_loss"]))

Loaded existing best QAT weights.
Epoch 1/10
400/400 [==============================] - ETA: 0s - loss: 0.0019 - mae: 0.0260
Epoch 1: val_loss improved from inf to 0.00129, saving model to /content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_final_best.weights.h5
400/400 [==============================] - 200s 492ms/step - loss: 0.0019 - mae: 0.0260 - val_loss: 0.0013 - val_mae: 0.0214 - lr: 1.0000e-05
Epoch 2/10
400/400 [==============================] - ETA: 0s - loss: 0.0019 - mae: 0.0251
Epoch 2: val_loss improved from 0.00129 to 0.00125, saving model to /content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_final_best.weights.h5
400/400 [==============================] - 196s 490ms/step - loss: 0.0019 - mae: 0.0251 - val_loss: 0.0013 - val_mae: 0.0208 - lr: 1.0000e-05
Epoch 3/10
400/400 [==============================] - ETA: 0s - loss: 0.0019 - mae: 0.0252
Epoch 3: val_loss improved from 0.00125 to 0.00124, saving model to /content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_final_best.w

In [22]:
import numpy as np
from pathlib import Path

# Existing arrays and models from our current session:
# restored_model = restored float T-MLP
# qat_model      = fine-tuned QAT T-MLP
# X_val_matched, Y_val_matched = matched validation windows

X_eval = np.asarray(X_val_matched, dtype=np.float32)
Y_eval = np.asarray(Y_val_matched, dtype=np.float32)

def calculate_metrics(reference, prediction):
    error = reference - prediction
    return {
        "MSE": float(np.mean(error ** 2)),
        "MAE": float(np.mean(np.abs(error))),
    }

# Noisy input baseline: output is simply the noisy signal.
noisy_metrics = calculate_metrics(Y_eval, X_eval)

# Float model.
float_predictions = restored_model.predict(
    X_eval, batch_size=256, verbose=0
)
float_metrics = calculate_metrics(Y_eval, float_predictions)

# Fine-tuned QAT model.
qat_predictions = qat_model.predict(
    X_eval, batch_size=256, verbose=0
)
qat_metrics = calculate_metrics(Y_eval, qat_predictions)

print("VALIDATION COMPARISON")
print("-" * 55)

for name, metrics in [
    ("Noisy input", noisy_metrics),
    ("Float T-MLP", float_metrics),
    ("QAT T-MLP", qat_metrics),
]:
    print(
        f"{name:15s} | "
        f"MSE: {metrics['MSE']:.8f} | "
        f"MAE: {metrics['MAE']:.8f}"
    )

print("\nQAT vs FLOAT")
mse_change = (
    (qat_metrics["MSE"] / float_metrics["MSE"]) - 1
) * 100
mae_change = (
    (qat_metrics["MAE"] / float_metrics["MAE"]) - 1
) * 100

print(f"MSE change: {mse_change:+.2f}%")
print(f"MAE change: {mae_change:+.2f}%")

print("\nMean absolute prediction difference:")
print(
    np.mean(np.abs(float_predictions - qat_predictions))
)

VALIDATION COMPARISON
-------------------------------------------------------
Noisy input     | MSE: 0.00256030 | MAE: 0.02096159
Float T-MLP     | MSE: 0.00123290 | MAE: 0.02033874
QAT T-MLP       | MSE: 0.00123593 | MAE: 0.02031623

QAT vs FLOAT
MSE change: +0.25%
MAE change: -0.11%

Mean absolute prediction difference:
0.0015976541


In [23]:
from pathlib import Path
import os
import tensorflow as tf
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
model_dir = project_dir / "saved_models"

print("=" * 68)
print("T-MLP QUANTIZED DEPLOYMENT PRE-FLIGHT")
print("=" * 68)

print("\n[1] SOFTWARE")
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

try:
    import tensorflow_model_optimization as tfmot
    print("TensorFlow Model Optimization:", tfmot.__version__)
except Exception as exc:
    print("TF-MOT import issue:", repr(exc))

print("\n[2] SAVED MODEL FILES")
if model_dir.exists():
    for p in sorted(model_dir.iterdir()):
        if p.is_file():
            print(f"{p.name:<42} {p.stat().st_size / 1024:10.2f} KB")
else:
    print("Model directory not found:", model_dir)

print("\n[3] LIVE MODEL OBJECTS")
for name in ("restored_model", "qat_model", "qat_base_model"):
    obj = globals().get(name)
    print(f"{name}: {'available' if obj is not None else 'missing'}")
    if obj is not None and hasattr(obj, "input_shape"):
        print("  input shape:", obj.input_shape)
        print("  output shape:", obj.output_shape)
        try:
            print("  parameters:", obj.count_params())
        except Exception:
            pass

print("\n[4] QAT LAYER SUMMARY")
if globals().get("qat_model") is not None:
    qat_model.summary()
else:
    print("qat_model is not in this runtime.")

print("\n[5] TFLITE CONVERTER")
try:
    converter = tf.lite.TFLiteConverter
    print("TensorFlow Lite converter: available")
except Exception as exc:
    print("TensorFlow Lite converter issue:", repr(exc))

print("\nPre-flight inspection complete. No training or checkpoint changes made.")


T-MLP QUANTIZED DEPLOYMENT PRE-FLIGHT

[1] SOFTWARE
TensorFlow: 2.21.0
NumPy: 2.1.3
TensorFlow Model Optimization: 0.8.1

[2] SAVED MODEL FILES
GRU_best.keras                                 478.09 KB
MLP_best.keras                                3876.92 KB
TMLP_QAT_best.weights.h5                       392.95 KB
TMLP_QAT_best_before_final.weights.h5          392.95 KB
TMLP_QAT_final_best.weights.h5                 392.95 KB
TMLP_QAT_initial.weights.h5                    222.52 KB
TMLP_QAT_int8.tflite                            54.09 KB
TMLP_best.keras                                292.70 KB

[3] LIVE MODEL OBJECTS
restored_model: available
  input shape: (None, 256)
  output shape: (None, 256)
  parameters: 19137
qat_model: available
  input shape: (None, 256)
  output shape: (None, 256)
  parameters: 19175
qat_base_model: available
  input shape: (None, 256)
  output shape: (None, 256)
  parameters: 19137

[4] QAT LAYER SUMMARY
Model: "T_MLP_tfkeras"
________________________________

In [24]:

from pathlib import Path
import os
import tensorflow as tf
import numpy as np

project_dir = Path("/content/drive/MyDrive/T-MLP")
model_dir = project_dir / "saved_models"

print("=" * 68)
print("T-MLP QUANTIZED DEPLOYMENT PRE-FLIGHT")
print("=" * 68)

# 1. Software versions
print("\n[1] SOFTWARE")
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

try:
    import tensorflow_model_optimization as tfmot
    print("TF-MOT:", tfmot.__version__)
except Exception as exc:
    print("TF-MOT import issue:", repr(exc))

# 2. Checkpoint inventory
print("\n[2] SAVED MODEL FILES")
if model_dir.exists():
    for path in sorted(model_dir.iterdir()):
        if path.is_file():
            print(f"{path.name:<42} {path.stat().st_size / 1024:10.2f} KB")
else:
    print("Model directory not found:", model_dir)

# 3. Check model objects in the current runtime
print("\n[3] LIVE MODEL OBJECTS")
for name in ("restored_model", "qat_model", "qat_base_model"):
    model = globals().get(name)
    print(f"{name}: {'available' if model is not None else 'missing'}")
    if model is not None and hasattr(model, "input_shape"):
        print("  Input shape:", model.input_shape)
        print("  Output shape:", model.output_shape)
        try:
            print("  Parameters:", model.count_params())
        except Exception:
            pass

# 4. Inspect the QAT architecture
print("\n[4] QAT MODEL SUMMARY")
if globals().get("qat_model") is not None:
    qat_model.summary()
else:
    print("qat_model is missing from the current runtime.")

# 5. Check TFLite availability
print("\n[5] TENSORFLOW LITE")
try:
    print("TFLite converter:", tf.lite.TFLiteConverter)
except Exception as exc:
    print("TFLite issue:", repr(exc))

print("\nInspection complete.")
print("No training performed. No checkpoints modified.")


T-MLP QUANTIZED DEPLOYMENT PRE-FLIGHT

[1] SOFTWARE
TensorFlow: 2.21.0
NumPy: 2.1.3
TF-MOT: 0.8.1

[2] SAVED MODEL FILES
GRU_best.keras                                 478.09 KB
MLP_best.keras                                3876.92 KB
TMLP_QAT_best.weights.h5                       392.95 KB
TMLP_QAT_best_before_final.weights.h5          392.95 KB
TMLP_QAT_final_best.weights.h5                 392.95 KB
TMLP_QAT_initial.weights.h5                    222.52 KB
TMLP_QAT_int8.tflite                            54.09 KB
TMLP_best.keras                                292.70 KB

[3] LIVE MODEL OBJECTS
restored_model: available
  Input shape: (None, 256)
  Output shape: (None, 256)
  Parameters: 19137
qat_model: available
  Input shape: (None, 256)
  Output shape: (None, 256)
  Parameters: 19175
qat_base_model: available
  Input shape: (None, 256)
  Output shape: (None, 256)
  Parameters: 19137

[4] QAT MODEL SUMMARY
Model: "T_MLP_tfkeras"
_______________________________________________________

In [25]:

from pathlib import Path
import numpy as np
import tensorflow as tf

project_dir = Path("/content/drive/MyDrive/T-MLP")
model_dir = project_dir / "saved_models"

qat_weights_path = model_dir / "TMLP_QAT_final_best.weights.h5"
tflite_path = model_dir / "TMLP_QAT_int8.tflite"

assert "qat_model" in globals(), "qat_model is missing."
assert "X_train" in globals(), "X_train is missing."
assert qat_weights_path.exists(), f"Missing QAT weights: {qat_weights_path}"

# Ensure the current model uses the best saved QAT weights.
qat_model.load_weights(str(qat_weights_path))

# Use a representative subset of existing training windows.
# This is for activation-range calibration, not training.
rng = np.random.default_rng(2026)
sample_count = min(200, len(X_train))
sample_indices = rng.choice(len(X_train), size=sample_count, replace=False)
representative_samples = X_train[sample_indices].astype(np.float32)

def representative_dataset():
    for sample in representative_samples:
        yield [sample.reshape(1, 256)]

print("TensorFlow:", tf.__version__)
print("Representative samples:", len(representative_samples))
print("Input shape:", qat_model.input_shape)

# Configure full-integer TFLite conversion.
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("\nConverting to full-integer INT8 TFLite...")
tflite_model = converter.convert()

# Write to a new file; preserve all existing checkpoints.
tflite_path.write_bytes(tflite_model)

print("\nConversion successful.")
print("Saved:", tflite_path)
print(f"TFLite model size: {tflite_path.stat().st_size / 1024:.2f} KB")


TensorFlow: 2.21.0
Representative samples: 200
Input shape: (None, 256)

Converting to full-integer INT8 TFLite...


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



Conversion successful.
Saved: /content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite
TFLite model size: 54.09 KB


In [26]:

from pathlib import Path
from collections import Counter
import numpy as np
import tensorflow as tf

project_dir = Path("/content/drive/MyDrive/T-MLP")
tflite_path = project_dir / "saved_models" / "TMLP_QAT_int8.tflite"

assert tflite_path.exists(), f"Model not found: {tflite_path}"
assert "X_val" in globals() and "Y_val" in globals()
assert "restored_model" in globals()

# ------------------------------------------------------------
# 1. LOAD TFLITE MODEL
# ------------------------------------------------------------
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

print("=" * 68)
print("TFLITE INPUT / OUTPUT")
print("=" * 68)
print("Input shape:", input_detail["shape"])
print("Input dtype:", input_detail["dtype"])
print("Input quantization:", input_detail["quantization"])
print("Output shape:", output_detail["shape"])
print("Output dtype:", output_detail["dtype"])
print("Output quantization:", output_detail["quantization"])

# ------------------------------------------------------------
# 2. INSPECT INTERNAL TENSOR TYPES
# ------------------------------------------------------------
tensor_details = interpreter.get_tensor_details()
dtype_counts = Counter(str(t["dtype"]) for t in tensor_details)

print("\n" + "=" * 68)
print("INTERNAL TENSOR DTYPE COUNTS")
print("=" * 68)
for dtype, count in sorted(dtype_counts.items()):
    print(f"{dtype:<20} {count}")

# List operator types and available input/output tensor dtypes.
print("\n" + "=" * 68)
print("OPERATOR TYPES")
print("=" * 68)

try:
    ops = interpreter._get_ops_details()
    op_counts = Counter(op["op_name"] for op in ops)
    for op_name, count in sorted(op_counts.items()):
        print(f"{op_name:<32} {count}")
except Exception as exc:
    print("Could not inspect operator details:", repr(exc))

# ------------------------------------------------------------
# 3. PREPARE VALIDATION SAMPLES
# ------------------------------------------------------------
n_eval = min(1024, len(X_val))
x_eval = np.asarray(X_val[:n_eval], dtype=np.float32)
y_eval = np.asarray(Y_val[:n_eval], dtype=np.float32)

in_scale, in_zero = input_detail["quantization"]
out_scale, out_zero = output_detail["quantization"]

if input_detail["dtype"] == np.int8 and in_scale > 0:
    x_quant = np.clip(
        np.round(x_eval / in_scale + in_zero), -128, 127
    ).astype(np.int8)
elif input_detail["dtype"] == np.uint8 and in_scale > 0:
    x_quant = np.clip(
        np.round(x_eval / in_scale + in_zero), 0, 255
    ).astype(np.uint8)
elif input_detail["dtype"] == np.float32:
    x_quant = x_eval
else:
    raise RuntimeError(
        f"Unsupported input dtype/quantization: "
        f"{input_detail['dtype']}, {input_detail['quantization']}"
    )

# ------------------------------------------------------------
# 4. RUN TFLITE INFERENCE
# ------------------------------------------------------------
tflite_predictions = []

for i in range(n_eval):
    sample = x_quant[i:i+1]

    # Adapt input shape if the converter exported a different batch shape.
    if tuple(interpreter.get_input_details()[0]["shape"]) != tuple(sample.shape):
        interpreter.resize_tensor_input(
            input_detail["index"], sample.shape, strict=False
        )
        interpreter.allocate_tensors()
        input_detail = interpreter.get_input_details()[0]
        output_detail = interpreter.get_output_details()[0]

    interpreter.set_tensor(input_detail["index"], sample)
    interpreter.invoke()
    pred = interpreter.get_tensor(output_detail["index"])

    if output_detail["dtype"] == np.int8 or output_detail["dtype"] == np.uint8:
        scale, zero = output_detail["quantization"]
        if scale <= 0:
            raise RuntimeError("Output quantization scale is invalid.")
        pred = (pred.astype(np.float32) - zero) * scale
    else:
        pred = pred.astype(np.float32)

    tflite_predictions.append(pred.reshape(-1))

tflite_predictions = np.asarray(tflite_predictions, dtype=np.float32)

# ------------------------------------------------------------
# 5. FLOAT REFERENCE AND METRICS
# ------------------------------------------------------------
float_predictions = restored_model.predict(
    x_eval, batch_size=128, verbose=0
)

def metric_pair(target, prediction):
    error = target - prediction
    return float(np.mean(error**2)), float(np.mean(np.abs(error)))

float_mse, float_mae = metric_pair(y_eval, float_predictions)
tflite_mse, tflite_mae = metric_pair(y_eval, tflite_predictions)

prediction_difference = float(
    np.mean(np.abs(float_predictions - tflite_predictions))
)

mse_change = (
    (tflite_mse - float_mse) / float_mse * 100
    if float_mse > 0 else float("nan")
)

print("\n" + "=" * 68)
print(f"VALIDATION COMPARISON ({n_eval} windows)")
print("=" * 68)
print(f"{'Model':<22}{'MSE':>16}{'MAE':>16}")
print("-" * 54)
print(f"{'Float T-MLP':<22}{float_mse:>16.8f}{float_mae:>16.8f}")
print(f"{'TFLite model':<22}{tflite_mse:>16.8f}{tflite_mae:>16.8f}")
print(f"\nTFLite MSE change vs float: {mse_change:+.2f}%")
print(f"Mean absolute prediction difference: {prediction_difference:.8f}")

print("\nArtifact size:", f"{tflite_path.stat().st_size / 1024:.2f} KB")
print("Verification run complete.")


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLITE INPUT / OUTPUT
Input shape: [  1 256]
Input dtype: <class 'numpy.int8'>
Input quantization: (0.009766020812094212, -3)
Output shape: [  1 256]
Output dtype: <class 'numpy.int8'>
Output quantization: (0.004689747933298349, 1)

INTERNAL TENSOR DTYPE COUNTS
<class 'numpy.int32'> 77
<class 'numpy.int8'> 92

OPERATOR TYPES
ADD                              15
CONCATENATION                    7
DELEGATE                         24
FILL                             4
FULLY_CONNECTED                  7
GATHER                           14
MEAN                             4
MUL                              8
PACK                             11
REDUCE_PROD                      16
RESHAPE                          22
RSQRT                            2
SHAPE                            11
SQUARED_DIFFERENCE               2
STRIDED_SLICE                    4
SUB                              2
TRANSPOSE                        4

VALIDATION COMPARISON (1024 windows)
Model                            

In [27]:

import numpy as np
import tensorflow as tf
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/T-MLP")
tflite_path = project_dir / "saved_models" / "TMLP_QAT_int8.tflite"

# Use the same validation subset as the previous comparison.
n_eval = min(1024, len(X_val))
x_eval = np.asarray(X_val[:n_eval], dtype=np.float32)

# 1. In-memory QAT prediction (float input/output interface)
qat_pred = qat_model.predict(x_eval, batch_size=128, verbose=0)

# 2. TFLite INT8 prediction
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

in_scale, in_zero = input_detail["quantization"]
out_scale, out_zero = output_detail["quantization"]

tflite_preds = []

for sample in x_eval:
    # Quantize input using the exported model's input scale/zero point.
    if in_scale <= 0:
        raise RuntimeError("Invalid TFLite input quantization scale.")

    q_input = np.clip(
        np.round(sample.reshape(1, 256) / in_scale + in_zero),
        -128, 127
    ).astype(np.int8)

    interpreter.set_tensor(input_detail["index"], q_input)
    interpreter.invoke()
    q_output = interpreter.get_tensor(output_detail["index"])

    # Dequantize INT8 output back to float.
    q_output = (
        q_output.astype(np.float32) - out_zero
    ) * out_scale

    tflite_preds.append(q_output.reshape(-1))

tflite_preds = np.asarray(tflite_preds, dtype=np.float32)

def mse(a, b):
    return float(np.mean((a - b) ** 2))

print("=" * 64)
print("QAT-TO-TFLITE CONVERSION DIAGNOSTIC")
print("=" * 64)
print("QAT prediction MSE vs clean target:", f"{mse(Y_val[:n_eval], qat_pred):.8f}")
print("TFLite prediction MSE vs clean target:", f"{mse(Y_val[:n_eval], tflite_preds):.8f}")
print("QAT vs TFLite prediction MSE:", f"{mse(qat_pred, tflite_preds):.8f}")
print(
    "Mean absolute prediction difference:",
    f"{np.mean(np.abs(qat_pred - tflite_preds)):.8f}"
)
print(
    "Maximum absolute prediction difference:",
    f"{np.max(np.abs(qat_pred - tflite_preds)):.8f}"
)


QAT-TO-TFLITE CONVERSION DIAGNOSTIC
QAT prediction MSE vs clean target: 0.00113489
TFLite prediction MSE vs clean target: 0.00136394
QAT vs TFLite prediction MSE: 0.00026900
Mean absolute prediction difference: 0.01375499
Maximum absolute prediction difference: 0.03751799


In [28]:

import tensorflow as tf
from pathlib import Path
from collections import Counter

tflite_path = Path(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

details = interpreter.get_tensor_details()
ops = interpreter._get_ops_details()

dtype_counts = Counter(str(t["dtype"]) for t in details)
op_counts = Counter(op["op_name"] for op in ops)

print("=" * 60)
print("TFLITE GRAPH DIAGNOSTIC")
print("=" * 60)

print("\nTensor dtypes:")
for dtype, count in sorted(dtype_counts.items()):
    print(f"{dtype:<20} {count}")

print("\nOperators:")
for name, count in sorted(op_counts.items()):
    print(f"{name:<30} {count}")

print("\nFloat32 tensors:")
float_tensors = [
    (t["index"], t["name"], t["shape"])
    for t in details
    if t["dtype"] == tf.float32
]

if float_tensors:
    for index, name, shape in float_tensors:
        print(f"Index {index}: {name}, shape={shape}")
else:
    print("None found in the exposed tensor details.")

print("\nDiagnostic complete.")


TFLITE GRAPH DIAGNOSTIC

Tensor dtypes:
<class 'numpy.int32'> 77
<class 'numpy.int8'> 92

Operators:
ADD                            15
CONCATENATION                  7
DELEGATE                       24
FILL                           4
FULLY_CONNECTED                7
GATHER                         14
MEAN                           4
MUL                            8
PACK                           11
REDUCE_PROD                    16
RESHAPE                        22
RSQRT                          2
SHAPE                          11
SQUARED_DIFFERENCE             2
STRIDED_SLICE                  4
SUB                            2
TRANSPOSE                      4

Float32 tensors:
None found in the exposed tensor details.

Diagnostic complete.


In [29]:

import time
import numpy as np
import tensorflow as tf
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/T-MLP")
tflite_path = project_dir / "saved_models" / "TMLP_QAT_int8.tflite"

assert tflite_path.exists()

# Fixed batch-one input, matching the intended streaming window.
x_bench = np.asarray(X_val[0:1], dtype=np.float32)

interpreter = tf.lite.Interpreter(
    model_path=str(tflite_path),
    num_threads=1
)
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
scale, zero = inp["quantization"]

x_int8 = np.clip(
    np.round(x_bench / scale + zero), -128, 127
).astype(np.int8)

# Warm-up
for _ in range(50):
    interpreter.set_tensor(inp["index"], x_int8)
    interpreter.invoke()

# Timed inference
iterations = 1000
start = time.perf_counter()

for _ in range(iterations):
    interpreter.set_tensor(inp["index"], x_int8)
    interpreter.invoke()

elapsed = time.perf_counter() - start
latency_ms = elapsed * 1000 / iterations

print("=" * 60)
print("INT8 TFLITE CPU LATENCY")
print("=" * 60)
print("Batch size: 1")
print("Window length: 256 samples")
print("Threads: 1")
print("Warm-up runs: 50")
print("Timed runs:", iterations)
print(f"Total time: {elapsed:.4f} s")
print(f"Mean latency: {latency_ms:.4f} ms/window")
print(f"Throughput: {1000 / latency_ms:.2f} windows/s")
print(f"Model size: {tflite_path.stat().st_size / 1024:.2f} KB")


INT8 TFLITE CPU LATENCY
Batch size: 1
Window length: 256 samples
Threads: 1
Warm-up runs: 50
Timed runs: 1000
Total time: 1.0391 s
Mean latency: 1.0391 ms/window
Throughput: 962.38 windows/s
Model size: 54.09 KB


In [32]:

import numpy as np
import tensorflow as tf
from pathlib import Path

# Use the same validation subset as the previous TFLite evaluation
N = 1024
X = np.asarray(X_val_matched[:N], dtype=np.float32)
Y = np.asarray(Y_val_matched[:N], dtype=np.float32)

tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

assert Path(tflite_path).exists(), f"Missing file: {tflite_path}"
assert X.shape == Y.shape and X.shape[1] == 256

# 1. Reference predictions
float_pred = np.asarray(
    restored_model(X, training=False)
).reshape(X.shape)

qat_pred = np.asarray(
    qat_model(X, training=False)
).reshape(X.shape)

# 2. Load the exported TFLite model
interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    num_threads=1
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

in_scale, in_zero = input_info["quantization"]
out_scale, out_zero = output_info["quantization"]

print("Input details:")
print("  dtype:", input_info["dtype"])
print("  scale:", in_scale, "zero point:", in_zero)
print("Output details:")
print("  dtype:", output_info["dtype"])
print("  scale:", out_scale, "zero point:", out_zero)

# 3. Run TFLite inference using the model's actual quantization parameters
tflite_pred = []

for x in X:
    x = x.reshape(1, 256)

    if input_info["dtype"] == np.int8:
        qmin, qmax = -128, 127
        x_input = np.clip(
            np.round(x / in_scale + in_zero), qmin, qmax
        ).astype(np.int8)
    elif input_info["dtype"] == np.uint8:
        qmin, qmax = 0, 255
        x_input = np.clip(
            np.round(x / in_scale + in_zero), qmin, qmax
        ).astype(np.uint8)
    else:
        x_input = x.astype(input_info["dtype"])

    interpreter.set_tensor(input_info["index"], x_input)
    interpreter.invoke()
    y_out = interpreter.get_tensor(output_info["index"])

    if output_info["dtype"] in (np.int8, np.uint8):
        y_out = (
            y_out.astype(np.float32) - out_zero
        ) * out_scale
    else:
        y_out = y_out.astype(np.float32)

    tflite_pred.append(y_out.reshape(256))

tflite_pred = np.asarray(tflite_pred, dtype=np.float32)

# 4. Compare errors
def metrics(pred, target):
    return (
        float(np.mean((pred - target) ** 2)),
        float(np.mean(np.abs(pred - target)))
    )

print("\n" + "=" * 58)
print("EXPORT DIAGNOSTICS — SAME VALIDATION WINDOWS")
print("=" * 58)

for name, pred in [
    ("Float T-MLP", float_pred),
    ("In-memory QAT", qat_pred),
    ("Exported TFLite", tflite_pred),
]:
    mse, mae = metrics(pred, Y)
    print(f"{name:20s} MSE={mse:.8f} | MAE={mae:.8f}")

print("\nPrediction disagreement:")
print(
    "QAT vs float — mean absolute difference:",
    f"{np.mean(np.abs(qat_pred - float_pred)):.8f}"
)
print(
    "TFLite vs QAT — mean absolute difference:",
    f"{np.mean(np.abs(tflite_pred - qat_pred)):.8f}"
)
print(
    "TFLite vs float — mean absolute difference:",
    f"{np.mean(np.abs(tflite_pred - float_pred)):.8f}"
)


print("\nInput clipping:")

quantized_input = X / in_scale + in_zero

clipping_fraction = np.mean(
    (quantized_input < -128) | (quantized_input > 127)
)

print(
    "Fraction of input values outside representable INT8 range:",
    f"{clipping_fraction:.4%}"
)

Input details:
  dtype: <class 'numpy.int8'>
  scale: 0.009766020812094212 zero point: -3
Output details:
  dtype: <class 'numpy.int8'>
  scale: 0.004689747933298349 zero point: 1

EXPORT DIAGNOSTICS — SAME VALIDATION WINDOWS
Float T-MLP          MSE=0.00113895 | MAE=0.02039487
In-memory QAT        MSE=0.00113489 | MAE=0.02036709
Exported TFLite      MSE=0.00136394 | MAE=0.02422140

Prediction disagreement:
QAT vs float — mean absolute difference: 0.00160729
TFLite vs QAT — mean absolute difference: 0.01375499
TFLite vs float — mean absolute difference: 0.01395321

Input clipping:
Fraction of input values outside representable INT8 range: 0.0084%


In [33]:

import tensorflow as tf
import numpy as np

tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    experimental_preserve_all_tensors=True
)
interpreter.allocate_tensors()

tensor_details = interpreter.get_tensor_details()

print("=" * 75)
print("TFLITE TENSOR QUANTIZATION AUDIT")
print("=" * 75)

dtype_counts = {}
for tensor in tensor_details:
    dtype_name = np.dtype(tensor["dtype"]).name
    dtype_counts[dtype_name] = dtype_counts.get(dtype_name, 0) + 1

print("\nTensor dtype counts:")
for dtype_name, count in sorted(dtype_counts.items()):
    print(f"{dtype_name:10s}: {count}")

print("\nQuantization parameters:")
print("Name | Shape | Dtype | Scale count | Scale range | Zero-point range")

for tensor in tensor_details:
    scales = tensor["quantization_parameters"]["scales"]
    zero_points = tensor["quantization_parameters"]["zero_points"]

    if len(scales) > 0:
        scale_summary = (
            f"{np.min(scales):.3g} to {np.max(scales):.3g}"
        )
        zp_summary = (
            f"{np.min(zero_points)} to {np.max(zero_points)}"
        )
    else:
        scale_summary = "NONE"
        zp_summary = "NONE"

    print(
        f"{tensor['name'][:42]:42s} | "
        f"{str(tensor['shape']):16s} | "
        f"{np.dtype(tensor['dtype']).name:7s} | "
        f"{len(scales):11d} | "
        f"{scale_summary:20s} | {zp_summary}"
    )

TFLITE TENSOR QUANTIZATION AUDIT

Tensor dtype counts:
int32     : 77
int8      : 92

Quantization parameters:
Name | Shape | Dtype | Scale count | Scale range | Zero-point range
serving_default_noisy_audio:0              | [  1 256]        | int8    |           1 | 0.00977 to 0.00977   | -3 to -3
arith.constant4                            | [1]              | int32   |           0 | NONE                 | NONE
arith.constant3                            | [1]              | int32   |           0 | NONE                 | NONE
arith.constant5                            | []               | int32   |           0 | NONE                 | NONE
arith.constant6                            | []               | int32   |           0 | NONE                 | NONE
arith.constant                             | [2]              | int32   |           0 | NONE                 | NONE
arith.constant2                            | [1]              | int32   |           0 | NONE                 | NONE
arith

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:465: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


In [34]:

import numpy as np
import tensorflow as tf

tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    experimental_preserve_all_tensors=True
)
interpreter.allocate_tensors()

tensor_details = interpreter.get_tensor_details()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

x = np.asarray(X_val_matched[0:1], dtype=np.float32)
scale, zero = input_info["quantization"]

xq = np.clip(
    np.round(x / scale + zero), -128, 127
).astype(np.int8)

interpreter.set_tensor(input_info["index"], xq)
interpreter.invoke()

# Show internal tensors with the largest absolute values.
# This is a diagnostic snapshot, not a direct comparison with
# QAT activations, because the two runtimes may use different layouts.
ranked = []

for detail in tensor_details:
    if detail["dtype"] not in (np.int8, np.uint8):
        continue

    try:
        value = interpreter.get_tensor(detail["index"])
        if value.size == 0:
            continue

        q = detail["quantization_parameters"]
        scales = q["scales"]
        zeros = q["zero_points"]

        if len(scales) == 1:
            real_value = (
                value.astype(np.float32) - zeros[0]
            ) * scales[0]
            max_abs = float(np.max(np.abs(real_value)))
            saturation = float(np.mean(
                (value == -128) | (value == 127)
            ))
            ranked.append((
                max_abs, saturation, detail["name"],
                detail["shape"], scales[0]
            ))
    except (ValueError, RuntimeError):
        pass

ranked.sort(reverse=True)

print("Largest internal INT8 activations:")
for max_abs, saturation, name, shape, scale in ranked[:20]:
    print(
        f"{name[:55]:55s} "
        f"shape={str(shape):15s} "
        f"max_abs={max_abs:.5f} "
        f"saturation={saturation:.2%} "
        f"scale={scale:.5g}"
    )

print("\nTFLite output quantization:", output_info["quantization"])


Largest internal INT8 activations:
T_MLP_tfkeras/temporal_mlp_block_2_layer_norm/FusedBatc shape=[ 1 64  1  1]   max_abs=82.45316 saturation=89.03% scale=0.43858
T_MLP_tfkeras/temporal_mlp_block_2_residual_add/add     shape=[  1 256  64]   max_abs=9.21707 saturation=0.00% scale=0.080852
T_MLP_tfkeras/temporal_mlp_block_2_layer_norm/Reshape   shape=[ 1  1 64  1]   max_abs=9.21707 saturation=0.00% scale=0.080852
T_MLP_tfkeras/temporal_mlp_block_2_layer_norm/FusedBatc shape=[ 1 64  1  1]   max_abs=9.21707 saturation=0.00% scale=0.080852
T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/add       shape=[  1 256  64]   max_abs=8.47326 saturation=0.00% scale=0.073045
T_MLP_tfkeras/quant_temporal_mlp_block_2_dense_1/Tensor shape=[1 1]           max_abs=8.47326 saturation=0.00% scale=0.073045
T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/mul       shape=[  1 256  64]   max_abs=7.88890 saturation=0.00% scale=0.073045
T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatc shape=[ 1 64  1  1]   max_

In [35]:

# Inspect the original float model's normalization layers.
# Read-only: this does not change or save the model.

for layer in restored_model.layers:
    if "temporal_mlp_block" in layer.name.lower():
        print(f"\nLayer: {layer.name} | Type: {type(layer).__name__}")

        if isinstance(layer, tf.keras.layers.LayerNormalization):
            print("  axis:", layer.axis)
            print("  epsilon:", layer.epsilon)
            print("  center:", layer.center)
            print("  scale:", layer.scale)

            weights = layer.get_weights()
            for i, w in enumerate(weights):
                print(
                    f"  weight[{i}]: shape={w.shape}, "
                    f"min={np.min(w):.6f}, max={np.max(w):.6f}, "
                    f"mean={np.mean(w):.6f}"
                )



Layer: temporal_mlp_block_1 | Type: TemporalMLPBlockRestore

Layer: temporal_mlp_block_2 | Type: TemporalMLPBlockRestore


In [37]:
# Read-only diagnostic: inspect nested LayerNormalization layers.

def inspect_nested_layers(layer, parent_name=""):
    current_name = f"{parent_name}/{layer.name}" if parent_name else layer.name

    if layer.__class__.__name__ == "LayerNormalization":
        print(f"\nLayer: {current_name}")
        print("  Type:", type(layer))
        print("  Axis:", layer.axis)
        print("  Epsilon:", layer.epsilon)
        print("  Center:", layer.center)
        print("  Scale:", layer.scale)

        for i, weight in enumerate(layer.get_weights()):
            print(
                f"  Weight[{i}]: shape={weight.shape}, "
                f"min={weight.min():.6f}, "
                f"max={weight.max():.6f}, "
                f"mean={weight.mean():.6f}"
            )

    # Explore layers nested inside custom blocks.
    if hasattr(layer, "layers"):
        for child in layer.layers:
            inspect_nested_layers(child, current_name)


for layer in restored_model.layers:
    inspect_nested_layers(layer)

In [38]:
# Inspect the custom block structure without changing anything.

for layer in restored_model.layers:
    if "temporal_mlp_block" in layer.name.lower():
        print(f"\n{'=' * 60}")
        print("Block:", layer.name)
        print("Class:", type(layer))
        print("MRO:", type(layer).__mro__)
        print("Has .layers:", hasattr(layer, "layers"))
        print("Has ._layers:", hasattr(layer, "_layers"))

        if hasattr(layer, "_layers"):
            print("Internal _layers:")
            for child in layer._layers:
                print(
                    "  ",
                    getattr(child, "name", "<unnamed>"),
                    type(child).__name__
                )

        print("Weights:")
        for weight in layer.weights:
            print(
                "  ",
                weight.name,
                tuple(weight.shape)
            )


Block: temporal_mlp_block_1
Class: <class '__main__.TemporalMLPBlockRestore'>
MRO: (<class '__main__.TemporalMLPBlockRestore'>, <class 'keras.src.layers.layer.Layer'>, <class 'keras.src.backend.tensorflow.layer.TFLayer'>, <class 'keras.src.backend.tensorflow.trackable.KerasAutoTrackable'>, <class 'tensorflow.python.trackable.autotrackable.AutoTrackable'>, <class 'tensorflow.python.trackable.base.Trackable'>, <class 'keras.src.ops.operation.Operation'>, <class 'keras.src.saving.keras_saveable.KerasSaveable'>, <class 'object'>)
Has .layers: False
Has ._layers: True
Internal _layers:
   dense_1 Dense
   dropout Dropout
   dense_2 Dense
   residual_add Add
   layer_norm LayerNormalization
Weights:
   kernel (64, 64)
   bias (64,)
   kernel (64, 64)
   bias (64,)
   gamma (64,)
   beta (64,)

Block: temporal_mlp_block_2
Class: <class '__main__.TemporalMLPBlockRestore'>
MRO: (<class '__main__.TemporalMLPBlockRestore'>, <class 'keras.src.layers.layer.Layer'>, <class 'keras.src.backend.tensor

In [39]:
# Read-only diagnostic — inspect nested LayerNormalization layers.

for block in restored_model.layers:
    if "temporal_mlp_block" not in block.name.lower():
        continue

    for child in block._layers:
        if child.__class__.__name__ == "LayerNormalization":
            print(f"\nBlock: {block.name}")
            print(f"Layer: {child.name}")
            print(f"Axis: {child.axis}")
            print(f"Epsilon: {child.epsilon}")
            print(f"Center: {child.center}")
            print(f"Scale: {child.scale}")

            for name, variable in zip(
                ["gamma", "beta"], child.weights
            ):
                values = variable.numpy()
                print(
                    f"{name}: shape={values.shape}, "
                    f"min={values.min():.6f}, "
                    f"max={values.max():.6f}, "
                    f"mean={values.mean():.6f}"
                )


Block: temporal_mlp_block_1
Layer: layer_norm
Axis: [-1]
Epsilon: 0.001
Center: True
Scale: True
gamma: shape=(64,), min=-0.140423, max=1.739613, mean=0.225521
beta: shape=(64,), min=-0.586417, max=0.538722, mean=-0.045598

Block: temporal_mlp_block_2
Layer: layer_norm
Axis: [-1]
Epsilon: 0.001
Center: True
Scale: True
gamma: shape=(64,), min=-0.037068, max=0.950353, mean=0.092570
beta: shape=(64,), min=-0.674498, max=0.685140, mean=-0.007182


In [40]:
import numpy as np
import tensorflow as tf

# Load the existing exported INT8 model; do not reconvert or overwrite it.
tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    num_threads=1
)
interpreter.allocate_tensors()

input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

X = np.asarray(X_val_matched[:1024], dtype=np.float32)
Y = np.asarray(Y_val_matched[:1024], dtype=np.float32)

# Reference predictions from the in-memory QAT model.
qat_pred = np.asarray(
    qat_model.predict(X, verbose=0),
    dtype=np.float32
).reshape(Y.shape)

# Run the exported INT8 model with explicit quantization.
in_scale, in_zero = input_info["quantization"]
out_scale, out_zero = output_info["quantization"]

if in_scale <= 0 or out_scale <= 0:
    raise ValueError("Unexpected quantization scale; inspect tensor details.")

tflite_pred = []

for sample in X:
    quantized_input = np.round(sample / in_scale + in_zero)
    quantized_input = np.clip(
        quantized_input, -128, 127
    ).astype(input_info["dtype"])

    interpreter.set_tensor(
        input_info["index"],
        quantized_input.reshape(input_info["shape"])
    )
    interpreter.invoke()

    raw_output = interpreter.get_tensor(output_info["index"])
    dequantized_output = (
        raw_output.astype(np.float32) - out_zero
    ) * out_scale

    tflite_pred.append(dequantized_output.reshape(Y.shape[1:]))

tflite_pred = np.asarray(tflite_pred, dtype=np.float32).reshape(Y.shape)

def report(name, prediction):
    print(
        f"{name}: "
        f"MSE={np.mean((prediction - Y) ** 2):.8f} | "
        f"MAE={np.mean(np.abs(prediction - Y)):.8f} | "
        f"Pred range=[{prediction.min():.5f}, {prediction.max():.5f}]"
    )

print("Input shape:", input_info["shape"])
print("Input dtype:", input_info["dtype"])
print("Input quantization:", input_info["quantization"])
print("Output shape:", output_info["shape"])
print("Output dtype:", output_info["dtype"])
print("Output quantization:", output_info["quantization"])

print("\nPrediction comparison")
report("In-memory QAT", qat_pred)
report("Exported TFLite INT8", tflite_pred)

print("\nExport discrepancy")
print(
    "Mean absolute QAT–TFLite difference:",
    np.mean(np.abs(qat_pred - tflite_pred))
)
print(
    "Maximum absolute QAT–TFLite difference:",
    np.max(np.abs(qat_pred - tflite_pred))
)

Input shape: [  1 256]
Input dtype: <class 'numpy.int8'>
Input quantization: (0.009766020812094212, -3)
Output shape: [  1 256]
Output dtype: <class 'numpy.int8'>
Output quantization: (0.004689747933298349, 1)

Prediction comparison
In-memory QAT: MSE=0.00113489 | MAE=0.02036709 | Pred range=[-0.26263, 0.24387]
Exported TFLite INT8: MSE=0.00136394 | MAE=0.02422140 | Pred range=[-0.25325, 0.23918]

Export discrepancy
Mean absolute QAT–TFLite difference: 0.013754989
Maximum absolute QAT–TFLite difference: 0.03751799


In [41]:
# Inspect the exported TFLite operator sequence.
# This does not modify the model.

ops = interpreter._get_ops_details()

print("Total operators:", len(ops))
print("\nOperator sequence:")

for i, op in enumerate(ops):
    print(
        f"{i:3d}. {op['op_name']:<25} "
        f"inputs={op['inputs']} "
        f"outputs={op['outputs']}"
    )

Total operators: 157

Operator sequence:
  0. SHAPE                     inputs=[0] outputs=[36]
  1. STRIDED_SLICE             inputs=[36  1  2  2] outputs=[37]
  2. PACK                      inputs=[37  3  4] outputs=[38]
  3. RESHAPE                   inputs=[ 0 38] outputs=[39]
  4. SHAPE                     inputs=[39] outputs=[40]
  5. GATHER                    inputs=[40  5] outputs=[41]
  6. REDUCE_PROD               inputs=[41  1] outputs=[42]
  7. CONCATENATION             inputs=[41  6] outputs=[43]
  8. GATHER                    inputs=[40  7] outputs=[44]
  9. REDUCE_PROD               inputs=[44  1] outputs=[45]
 10. PACK                      inputs=[42 45] outputs=[46]
 11. RESHAPE                   inputs=[ 0 46] outputs=[47]
 12. FULLY_CONNECTED           inputs=[47 35 -1] outputs=[48]
 13. RESHAPE                   inputs=[48 43] outputs=[49]
 14. ADD                       inputs=[49 34] outputs=[50]
 15. SHAPE                     inputs=[50] outputs=[51]
 16. GATHER  

In [2]:
import tensorflow as tf
import numpy as np
import os

# Path to the existing exported INT8 model
tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

if not os.path.exists(tflite_path):
    raise FileNotFoundError(
        f"TFLite model not found: {tflite_path}"
    )

# Recreate the interpreter
interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    num_threads=1
)
interpreter.allocate_tensors()

print("TFLite interpreter restored successfully.")
print("Input:", interpreter.get_input_details()[0]["quantization"])
print("Output:", interpreter.get_output_details()[0]["quantization"])

# Build tensor lookup
tensor_details = {
    t["index"]: t
    for t in interpreter.get_tensor_details()
}
ops = interpreter._get_ops_details()

# Inspect normalization arithmetic in both blocks
for start, end, name in [
    (47, 60, "Normalization block 1"),
    (93, 106, "Normalization block 2"),
]:
    print(f"\n{'=' * 65}")
    print(name)

    for i in range(start, end + 1):
        op = ops[i]
        print(f"\nOperator {i}: {op['op_name']}")

        for role, indices in [
            ("IN ", op["inputs"]),
            ("OUT", op["outputs"])
        ]:
            for idx in indices:
                if idx < 0 or idx not in tensor_details:
                    continue

                t = tensor_details[idx]
                q = t["quantization"]

                print(
                    f"  {role} {idx:3d} | "
                    f"{t['name']} | "
                    f"{t['dtype'].__name__} | "
                    f"scale={q[0]:.8g} | "
                    f"zero_point={q[1]}"
                )

TFLite interpreter restored successfully.
Input: (0.009766020812094212, -3)
Output: (0.004689747933298349, 1)

Normalization block 1

Operator 47: MEAN
  IN   82 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3 | int8 | scale=0.10425608 | zero_point=-16
  IN   10 | arith.constant10 | int32 | scale=0 | zero_point=0
  OUT  83 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV31 | int8 | scale=0.0036879554 | zero_point=-128

Operator 48: SQUARED_DIFFERENCE
  IN   82 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3 | int8 | scale=0.10425608 | zero_point=-16
  IN   83 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV31 | int8 | scale=0.0036879554 | zero_point=-128
  OUT  84 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV32 | int8 | scale=0.83001566 | zero_point=-128

Operator 49: MEAN
  IN   84 | T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV32 | int8 | scale=0.83001566 | zero_point=-128
  IN   10 | ar

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [4]:
import os

project_dir = "/content/drive/MyDrive/T-MLP"
validation_dir = os.path.join(project_dir, "processed_data", "validation")

print("Validation directory exists:", os.path.isdir(validation_dir))

if os.path.isdir(validation_dir):
    files = sorted(
        f for f in os.listdir(validation_dir)
        if f.endswith(".npz")
    )
    print("Validation NPZ files:", len(files))
    print("First 5 files:", files[:5])

Validation directory exists: True
Validation NPZ files: 100
First 5 files: ['sample_000000.npz', 'sample_000001.npz', 'sample_000002.npz', 'sample_000003.npz', 'sample_000004.npz']


In [1]:
import os
import numpy as np

validation_dir = "/content/drive/MyDrive/T-MLP/processed_data/validation"

X_val_list = []
Y_val_list = []

rng = np.random.default_rng(2026)

files = sorted(
    f for f in os.listdir(validation_dir)
    if f.endswith(".npz")
)

for filename in files:
    path = os.path.join(validation_dir, filename)

    with np.load(path, allow_pickle=False) as data:
        noisy = np.asarray(data["noisy_speech"], dtype=np.float32).reshape(-1)
        clean = np.asarray(data["clean_speech"], dtype=np.float32).reshape(-1)

    # Use complete, non-overlapping 256-sample windows.
    n_windows = min(len(noisy), len(clean)) // 256

    if n_windows == 0:
        continue

    count = min(n_windows, 64)
    starts = np.arange(n_windows) * 256

    if n_windows > count:
        chosen = np.sort(rng.choice(starts, size=count, replace=False))
    else:
        chosen = starts

    for start in chosen:
        X_val_list.append(noisy[start:start + 256])
        Y_val_list.append(clean[start:start + 256])

X_val_matched = np.asarray(X_val_list, dtype=np.float32)
Y_val_matched = np.asarray(Y_val_list, dtype=np.float32)

print("X_val_matched:", X_val_matched.shape)
print("Y_val_matched:", Y_val_matched.shape)

X_val_matched: (6400, 256)
Y_val_matched: (6400, 256)


In [2]:
import numpy as np
import tensorflow as tf

tflite_path = (
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_int8.tflite"
)

interpreter = tf.lite.Interpreter(
    model_path=tflite_path,
    experimental_preserve_all_tensors=True,
    num_threads=1
)
interpreter.allocate_tensors()

# Confirm X_val_matched is available after any runtime restart.
if "X_val_matched" not in globals():
    raise NameError(
        "X_val_matched is not in memory. Reload the validation windows first."
    )

X = np.asarray(X_val_matched[:100], dtype=np.float32)
input_info = interpreter.get_input_details()[0]
tensor_info = interpreter.get_tensor_details()

# Focus on the internal normalization tensors identified earlier.
targets = [
    t for t in tensor_info
    if "temporal_mlp_block_1_layer_norm/FusedBatchNormV3" in t["name"]
    or "temporal_mlp_block_2_layer_norm/FusedBatchNormV3" in t["name"]
]

stats = {
    t["index"]: {
        "name": t["name"],
        "count": 0,
        "min": np.inf,
        "max": -np.inf,
        "low": 0,
        "high": 0
    }
    for t in targets
    if t["dtype"] == np.int8
}

scale, zero = input_info["quantization"]

for sample in X:
    qx = np.clip(np.round(sample / scale + zero), -128, 127).astype(np.int8)
    interpreter.set_tensor(input_info["index"], qx.reshape(input_info["shape"]))
    interpreter.invoke()

    for idx, s in stats.items():
        a = interpreter.get_tensor(idx).astype(np.int16)
        s["count"] += a.size
        s["min"] = min(s["min"], int(a.min()))
        s["max"] = max(s["max"], int(a.max()))
        s["low"] += int(np.sum(a == -128))
        s["high"] += int(np.sum(a == 127))

print("Internal INT8 tensor statistics across", len(X), "windows")

for s in stats.values():
    n = s["count"]
    if n == 0:
        continue
    print(
        f"\n{s['name']}\n"
        f"  observed int8 range: [{s['min']}, {s['max']}]\n"
        f"  -128 frequency: {100*s['low']/n:.2f}%\n"
        f"  +127 frequency: {100*s['high']/n:.2f}%"
    )

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:465: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


Internal INT8 tensor statistics across 100 windows

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3
  observed int8 range: [-24, -3]
  -128 frequency: 0.00%
  +127 frequency: 0.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV31
  observed int8 range: [-73, -71]
  -128 frequency: 0.00%
  +127 frequency: 0.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV32
  observed int8 range: [-128, -126]
  -128 frequency: 96.88%
  +127 frequency: 0.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV33
  observed int8 range: [-127, -127]
  -128 frequency: 0.00%
  +127 frequency: 0.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV34
  observed int8 range: [-127, -127]
  -128 frequency: 0.00%
  +127 frequency: 0.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV35
  observed int8 range: [127, 127]
  -128 frequency: 0.00%
  +127 frequency: 100.00%

T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNor

In [3]:

# Inspect suspicious LayerNorm-related tensors and their graph connections.
# Uses the existing debugging interpreter; no retraining or reconversion.

tensor_details = interpreter.get_tensor_details()
ops = interpreter._get_ops_details()

name_by_index = {
    t["index"]: t["name"]
    for t in tensor_details
}

detail_by_index = {
    t["index"]: t
    for t in tensor_details
}

suspicious = [
    t for t in tensor_details
    if (
        "temporal_mlp_block_1_layer_norm/FusedBatchNormV3" in t["name"]
        or "temporal_mlp_block_2_layer_norm/FusedBatchNormV3" in t["name"]
    )
]

for t in suspicious:
    idx = t["index"]
    q = t.get("quantization_parameters", {})
    scales = q.get("scales", [])
    zero_points = q.get("zero_points", [])

    print("\n" + "=" * 75)
    print("Tensor:", t["name"])
    print("Index:", idx)
    print("Shape:", t["shape"])
    print("Dtype:", t["dtype"])
    print("Scale(s):", scales[:8])
    print("Zero point(s):", zero_points[:8])

    print("Related operators:")
    found = False

    for op_idx, op in enumerate(ops):
        inputs = list(op["inputs"])
        outputs = list(op["outputs"])

        if idx in inputs or idx in outputs:
            found = True
            direction = []
            if idx in inputs:
                direction.append("INPUT")
            if idx in outputs:
                direction.append("OUTPUT")

            print(
                f"  Op {op_idx}: {op['op_name']} "
                f"({', '.join(direction)})"
            )
            print("    Inputs:")
            for i in inputs:
                if i in detail_by_index:
                    d = detail_by_index[i]
                    print(
                        f"      {i}: {d['name']} | "
                        f"shape={d['shape']} | dtype={d['dtype']}"
                    )
            print("    Outputs:")
            for o in outputs:
                if o in detail_by_index:
                    d = detail_by_index[o]
                    print(
                        f"      {o}: {d['name']} | "
                        f"shape={d['shape']} | dtype={d['dtype']}"
                    )

    if not found:
        print("  No directly connected operators found.")



Tensor: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3
Index: 82
Shape: [  1  64   1 256]
Dtype: <class 'numpy.int8'>
Scale(s): [0.10425608]
Zero point(s): [-16]
Related operators:
  Op 46: TRANSPOSE (OUTPUT)
    Inputs:
      81: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/Reshape | shape=[  1 256  64   1] | dtype=<class 'numpy.int8'>
      9: arith.constant12 | shape=[4] | dtype=<class 'numpy.int32'>
    Outputs:
      82: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3 | shape=[  1  64   1 256] | dtype=<class 'numpy.int8'>
  Op 47: MEAN (INPUT)
    Inputs:
      82: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV3 | shape=[  1  64   1 256] | dtype=<class 'numpy.int8'>
      10: arith.constant10 | shape=[3] | dtype=<class 'numpy.int32'>
    Outputs:
      83: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/FusedBatchNormV31 | shape=[256] | dtype=<class 'numpy.int8'>
  Op 48: SQUARED_DIFFERENCE (INPUT)
    Inputs:
      82: T_MLP_tfkeras/t

In [4]:

import numpy as np

# Indices identified from the graph:
# 94  = output of the first LayerNorm calculation
# 140 = output of the second LayerNorm calculation

check_indices = [94, 140]
stats = {
    idx: {
        "min": np.inf,
        "max": -np.inf,
        "count": 0,
        "low": 0,
        "high": 0,
        "name": None,
        "scale": None,
        "zero_point": None,
    }
    for idx in check_indices
}

for t in interpreter.get_tensor_details():
    if t["index"] in stats:
        q = t["quantization_parameters"]
        stats[t["index"]]["name"] = t["name"]
        stats[t["index"]]["scale"] = q["scales"]
        stats[t["index"]]["zero_point"] = q["zero_points"]

for sample in X_val_matched[:100]:
    input_info = interpreter.get_input_details()[0]
    scale, zero = input_info["quantization"]

    qx = np.clip(
        np.round(sample / scale + zero), -128, 127
    ).astype(np.int8)

    interpreter.set_tensor(
        input_info["index"],
        qx.reshape(input_info["shape"])
    )
    interpreter.invoke()

    for idx in check_indices:
        a = interpreter.get_tensor(idx).astype(np.int16)
        s = stats[idx]

        s["min"] = min(s["min"], int(a.min()))
        s["max"] = max(s["max"], int(a.max()))
        s["count"] += a.size
        s["low"] += int(np.sum(a == -128))
        s["high"] += int(np.sum(a == 127))

for idx, s in stats.items():
    print("\n" + "=" * 60)
    print("Tensor:", s["name"])
    print("Scale:", s["scale"])
    print("Zero point:", s["zero_point"])
    print("Integer range:", (s["min"], s["max"]))
    print("At -128 (%):", round(100 * s["low"] / s["count"], 3))
    print("At +127 (%):", round(100 * s["high"] / s["count"], 3))


Tensor: T_MLP_tfkeras/temporal_mlp_block_1_layer_norm/Reshape_1
Scale: [0.04213561]
Zero point: [-10]
Integer range: (-118, 110)
At -128 (%): 0.0
At +127 (%): 0.0

Tensor: T_MLP_tfkeras/temporal_mlp_block_2_layer_norm/Reshape_1
Scale: [0.04554477]
Zero point: [-2]
Integer range: (-117, 113)
At -128 (%): 0.0
At +127 (%): 0.0


In [6]:

import os

project_dir = "/content/drive/MyDrive/T-MLP"
models_dir = os.path.join(project_dir, "saved_models")

for filename in [
    "TMLP_QAT_final_best.weights.h5",
    "TMLP_best.keras",
    "TMLP_QAT_int8.tflite",
]:
    path = os.path.join(models_dir, filename)
    print(
        f"{filename}:",
        "EXISTS" if os.path.isfile(path) else "NOT FOUND",
        f"({os.path.getsize(path) / 1024:.1f} KB)" if os.path.isfile(path) else ""
    )


TMLP_QAT_final_best.weights.h5: EXISTS (392.9 KB)
TMLP_best.keras: EXISTS (292.7 KB)
TMLP_QAT_int8.tflite: EXISTS (54.1 KB)


In [9]:

import tensorflow as tf
import tf_keras as tk
import tensorflow_model_optimization as tfmot

print("TensorFlow:", tf.__version__)
print("tf_keras:", tk.__version__)
print("TF-MOT:", tfmot.__version__)

print("QAT checkpoint exists:", os.path.isfile(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_QAT_final_best.weights.h5"
))
print("Float model exists:", os.path.isfile(
    "/content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras"
))


TensorFlow: 2.21.0
tf_keras: 2.21.0
TF-MOT: 0.8.1
QAT checkpoint exists: True
Float model exists: True


In [10]:
import numpy as np

# Use identical inputs for both models.
X = np.asarray(X_val_matched[:100], dtype=np.float32)
Y = np.asarray(Y_val_matched[:100], dtype=np.float32)

# In-memory QAT prediction
pred_qat = np.asarray(qat_model.predict(X, verbose=0))

# Exported INT8 TFLite prediction
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]

input_scale, input_zero = input_info["quantization"]
output_scale, output_zero = output_info["quantization"]

pred_tflite = []

for sample in X:
    qx = np.clip(
        np.round(sample / input_scale + input_zero),
        -128, 127
    ).astype(np.int8)

    interpreter.set_tensor(
        input_info["index"],
        qx.reshape(input_info["shape"])
    )
    interpreter.invoke()

    qy = interpreter.get_tensor(output_info["index"])
    y = (qy.astype(np.float32) - output_zero) * output_scale
    pred_tflite.append(y.reshape(-1))

pred_tflite = np.asarray(pred_tflite, dtype=np.float32)
pred_qat = pred_qat.reshape(pred_tflite.shape)
Y = Y.reshape(pred_tflite.shape)

def metrics(reference, prediction):
    return {
        "MSE": float(np.mean((reference - prediction) ** 2)),
        "MAE": float(np.mean(np.abs(reference - prediction))),
    }

print("QAT metrics:", metrics(Y, pred_qat))
print("TFLite metrics:", metrics(Y, pred_tflite))
print(
    "Mean absolute QAT–TFLite prediction difference:",
    float(np.mean(np.abs(pred_qat - pred_tflite)))
)
print(
    "Maximum absolute QAT–TFLite prediction difference:",
    float(np.max(np.abs(pred_qat - pred_tflite)))
)

NameError: name 'qat_model' is not defined

In [11]:
import os
import json
import zipfile

project_dir = "/content/drive/MyDrive/T-MLP"
model_path = os.path.join(
    project_dir, "saved_models", "TMLP_best.keras"
)

with zipfile.ZipFile(model_path, "r") as archive:
    print("Archive files:", archive.namelist())

    config = json.loads(archive.read("config.json"))

print("\nModel class:", config.get("class_name"))
print("Model name:", config.get("config", {}).get("name"))

print("\nLayers and configurations:")
model_config = config.get("config", {})
layers = model_config.get("layers", [])

for i, layer in enumerate(layers):
    layer_config = layer.get("config", {})
    print(
        f"{i:02d}. {layer.get('class_name')} | "
        f"name={layer_config.get('name')} | "
        f"config={layer_config}"
    )

Archive files: ['metadata.json', 'config.json', 'model.weights.h5']

Model class: Functional
Model name: T_MLP

Layers and configurations:
00. InputLayer | name=noisy_audio | config={'batch_shape': [None, 256], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'noisy_audio', 'optional': False}
01. Reshape | name=temporal_reshape | config={'name': 'temporal_reshape', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'target_shape': [256, 1]}
02. Dense | name=feature_projection | config={'name': 'feature_projection', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 139473906225168}, 'units': 64, 'activation': 'relu', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {

In [42]:
import zipfile
import json

model_path = "/content/drive/MyDrive/T-MLP/saved_models/TMLP_best.keras"

with zipfile.ZipFile(model_path, "r") as archive:
    config = json.loads(archive.read("config.json"))
    metadata = json.loads(archive.read("metadata.json"))

print("Metadata:")
print(json.dumps(metadata, indent=2))

print("\nCustom layer configuration:")
for layer in config.get("config", {}).get("layers", []):
    if layer.get("class_name") == "TemporalMLPBlock":
        print(json.dumps(layer, indent=2))

Metadata:
{
  "keras_version": "3.13.2",
  "date_saved": "2026-10-06@16:08:27"
}

Custom layer configuration:
{
  "module": null,
  "class_name": "TemporalMLPBlock",
  "config": {
    "name": "temporal_mlp_block_1",
    "hidden_dim": 64,
    "dropout_rate": 0.1,
    "trainable": true,
    "dtype": {
      "module": "keras",
      "class_name": "DTypePolicy",
      "config": {
        "name": "float32"
      },
      "registered_name": null,
      "shared_object_id": 139473906225168
    }
  },
  "registered_name": "TemporalMLPBlock",
  "build_config": {
    "input_shape": [
      null,
      256,
      64
    ]
  },
  "name": "temporal_mlp_block_1",
  "inbound_nodes": [
    {
      "args": [
        {
          "class_name": "__keras_tensor__",
          "config": {
            "shape": [
              null,
              256,
              64
            ],
            "dtype": "float32",
            "keras_history": [
              "feature_projection",
              0,
              